# 08_skills 02 · 三种技能载体：本地目录 / 云端 Store / Claude Code

上一课（`01_概念与原理与SKILL示例.ipynb`）讲清了「技能是什么、SKILL.md 怎么写」。
这一课回答另一个问题：**同一份技能，可以存在哪里、由谁发现、怎么加载。**

| 载体 | 存哪 | 谁发现 | 怎么加载 | 适用场景 |
|---|---|---|---|---|
| ① `skills=[...]` | 宿主机目录（本课是 `Agent/08_skills/skills/`） | DeepAgents 的 `SkillsMiddleware` | 启动时只把 `name`+`description` 拼进 system prompt；模型要用时自己 `read_file` 拉全文 | 单机、单用户；技能跟着代码仓库走 |
| ② 云技能（Store） | LangGraph Store（本机 PostgreSQL） | 同一个 `SkillsMiddleware`，只是 backend 换成 `CompositeBackend`+`StoreBackend` | 写法一模一样（还是 `skills=["/skills/"]`），文件改从数据库里读 | 多用户：一套服务给所有人用，按 `namespace` 一人一份技能库 |
| ③ Claude Code | 宿主机目录：全局 `~/.claude/skills/` 或项目 `<项目>/.claude/skills/` | Claude Code / Cursor / Trae / OpenCode 客户端自己扫 | 启动时扫目录清单；**「安装」= 把技能目录复制过去**，没有注册表、没有配置文件 | 把技能装进真实客户端，给 IDE / CLI 用 |

三者的**文件格式是同一套**（Agent Skills 规范：YAML front-matter + Markdown 正文 + 可选的
`scripts/` `references/` `assets/`），区别只在「谁来发现」和「文件放在哪」。

> **本 notebook 由 `Agent/_py_source/08_skills/` 下 3 个脚本合并而成**：
> `03_deepagents技能_jxsd.py`（191 行有效代码）、`04_云技能_jxsd.py`（228 行）、
> `05_ClaudeCode技能_jxsd.py`（246 行）。代码一行没丢，散落的 `#` 讲解注释提升成了 markdown。

**官方文档**
- Skills（DeepAgents）：<https://docs.langchain.com/oss/python/deepagents/skills>
- Backends（`CompositeBackend` / `StoreBackend`）：<https://docs.langchain.com/oss/python/deepagents/backends>
- Stores（Store 的 namespace 与持久化）：<https://docs.langchain.com/oss/python/langgraph/stores>
- Claude Code Skills：<https://code.claude.com/docs/en/skills>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 本节 1.4 / 2.4 两格会真实调用 `.env` 里配置的大模型 |
| 依赖 | `deepagents` 0.7.13 / `langchain` / `langgraph`（本项目 venv 已装） |
| 密钥 | `settings.api_key` / `settings.base_url` / `settings.model_name`（已配置） |
| 前置服务 | 本机 PostgreSQL 17 —— 云技能小节用 `settings.pg_uri` 把 SKILL.md 存进 Store；不可用时该小节打印中文提示并**跳过**，notebook 不抛 traceback |
| 资源目录 | `Agent/08_skills/skills/` 与 `Agent/08_skills/.claude/skills/`（仓库已跟踪，本 notebook **只读引用**，不生成、不删除） |
| 预计耗时 | 约 60~120 秒（含 2 次真实模型调用） |

> 只有第 3 节（Claude Code）是纯标准库 + 只读列举，其余各节都需要上面这些条件。

## 本节地图

三种载体最终都汇到同一件事上：**模型手里只有「技能清单」，全文要自己去读**。

```mermaid
graph LR
    subgraph L1["① 本地目录"]
        A1["Agent/08_skills/skills/"]
    end
    subgraph L2["② 云端 Store"]
        B1["PostgreSQL langgraph.store<br/>namespace=(课程, 用户)"]
    end
    subgraph L3["③ Claude Code"]
        C1["~/.claude/skills/（全局）<br/>项目/.claude/skills/（项目级）"]
    end
    A1 --> M1["SkillsMiddleware<br/>sources=['/skills/']"]
    B1 --> M2["SkillsMiddleware<br/>backend=CompositeBackend"]
    C1 --> CP["客户端启动时扫描目录"]
    M1 --> P["system prompt 里只有<br/>name + description + 路径"]
    M2 --> P
    P --> R["模型自己 read_file 拉 SKILL.md 全文<br/>（渐进式披露）"]
    CP --> R
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 小节 | 载体 | 关键代码 | 一句话结论 |
|---|---|---|---|
| 1 | DeepAgents 本地技能 | `create_deep_agent(skills=["/skills/"], backend=LocalShellBackend(root_dir=...))` | `skills` 指向**技能的父目录**，不是技能目录、也不是 SKILL.md |
| 2 | 云技能（Store） | `CompositeBackend(default=StateBackend(), routes={"/skills/": StoreBackend(...)})` | 换 backend 不换写法；隔离粒度是 `namespace` |
| 3 | Claude Code | `shutil.copytree(技能目录, <项目>/.claude/skills/)` | 复制过去就算装好，重启客户端生效 |

**与上下节的衔接**：上一课手工做了 `skills/code-review-skill/`（含 `references/` 两份规范），
这一课就把它分别挂到三种载体上——本地目录（第 1 节）、数据库 Store（第 2 节）、
客户端安装路径（第 3 节）。下一课进入工具与函数调用。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课的第 1、2 节用到的 `skills/`、`.claude/` 资源目录都在自身所在目录下，
> 所以还要靠这一格给出的 `NB_DIR` 来定位（notebook 里没有 `__file__`）。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\08_skills\tmp_nb_work
```

### 前置条件自检

这一格把后面各小节需要的条件**一次性查完**，并给每个小节留一个开关（`READY`）。
少了任何一项，对应小节会打印中文提示而不是抛 traceback —— 教学 notebook 的目标是
「缺环境也能看完讲解」，不是「一步不对就全红」。

| 开关 | 需要的条件 | 用它的地方 |
|---|---|---|
| `READY["model"]` | `.env` 里的 `MODEL_NAME` / `API_KEY` / `BASE_URL` | 1.4、2.4 |
| `READY["deepagents"]` | `deepagents` 包（`SkillsMiddleware` 在它里面） | 1.2、1.3、3.5 |
| `READY["local_skill"]` | `Agent/08_skills/skills/code-review-skill/SKILL.md` | 1、3 |
| `READY["claude_skill"]` | `Agent/08_skills/.claude/skills/code-review-skill/SKILL.md` | 3 |
| `READY["pg"]` | `.env` 里的 `PG_URI`（本机 PostgreSQL 17） | 2 |

In [ ]:
# ===== 前置条件自检（缺什么就打印中文提示，后面的小节自己跳过）=====
import importlib.util

from config import settings

READY: dict[str, bool] = {}

# ① 模型：1.4 / 2.4 要真调
READY["model"] = bool(settings.model_name and settings.api_key and settings.base_url)
print(f"[{'✓' if READY['model'] else '×'}] 大模型：model={settings.model_name} base_url={settings.base_url} "
      f"api_key={'已配置' if settings.api_key else '缺失'}")
if not READY["model"]:
    print("    ！请在 F:\\ProGram\\Python_Base\\.env 里补上 MODEL_NAME / API_KEY / BASE_URL 后再运行")

# ② deepagents：技能发现全靠它的 SkillsMiddleware
READY["deepagents"] = importlib.util.find_spec("deepagents") is not None
print(f"[{'✓' if READY['deepagents'] else '×'}] deepagents："
      f"{'已安装' if READY['deepagents'] else '未安装（三节里的技能发现都要用它）'}")

# ③ 本地技能目录：上一课的产物，本仓库已跟踪
SKILL_MD = NB_DIR / "skills" / "code-review-skill" / "SKILL.md"
READY["local_skill"] = SKILL_MD.exists()
print(f"[{'✓' if READY['local_skill'] else '×'}] 本地技能（父目录 skills/）：{SKILL_MD}")

# ④ 项目级 Claude Code 安装目录：也是仓库里现成的，本课只读引用
CLAUDE_SKILL_MD = NB_DIR / ".claude" / "skills" / "code-review-skill" / "SKILL.md"
READY["claude_skill"] = CLAUDE_SKILL_MD.exists()
print(f"[{'✓' if READY['claude_skill'] else '×'}] 项目级 .claude 技能：{CLAUDE_SKILL_MD}")

# ⑤ PostgreSQL：云技能小节要把 SKILL.md 存进 Store
READY["pg"] = bool(getattr(settings, "pg_uri", ""))
_tail = settings.pg_uri.split("@")[-1] if READY["pg"] and "@" in settings.pg_uri else ""
print(f"[{'✓' if READY['pg'] else '×'}] PostgreSQL：PG_URI {'已配置' if READY['pg'] else '未配置'}"
      + (f"（{_tail}，用户名口令不打印）" if _tail else ""))

print()
print("结论：没打 ✓ 的项，对应小节会打印中文提示并跳过（notebook 不会抛 traceback）。")

### 预期输出

```text
[✓] 大模型：model=deepseek-flash base_url=https://api.deepseek.com api_key=已配置
[✓] deepagents：已安装
[✓] 本地技能（父目录 skills/）：F:\ProGram\Python_Base\Agent\08_skills\skills\code-review-skill\SKILL.md
[✓] 项目级 .claude 技能：F:\ProGram\Python_Base\Agent\08_skills\.claude\skills\code-review-skill\SKILL.md
[✓] PostgreSQL：PG_URI 已配置（127.0.0.1:5432/langgraph，用户名口令不打印）

结论：没打 ✓ 的项，对应小节会打印中文提示并跳过（notebook 不会抛 traceback）。
```

## 1. 课案原版：DeepAgents 的 `skills=[...]`

课案这一节给的例子是「把图片转成 mermaid」的技能，代码只有十几行：

```python
agent = create_deep_agent(
    model=model,
    backend=LocalShellBackend(root_dir=".", inherit_env=True),
    skills=["/"],  # 扫描根目录 -> 发现 /image-to-mermaid/SKILL.md
)
```

课案原文里 `skills` 那行的注释被截断了（只留下「# 注意skills 指向技能的【父目」）。
本小节把它补全，并逐行解释这个参数的**三个坑**：

| # | 坑 | 正确写法 |
|---|---|---|
| 1 | `skills` 指向**技能的父目录** | `skills=["/skills/"]` —— 装着若干技能目录的那一层 |
| 2 | 路径是 **backend 的虚拟路径**，不是宿主机绝对路径 | 相对于 `LocalShellBackend(root_dir=...)` 那一层；换 root_dir 就要换这个路径 |
| 3 | 传了 `skills` 就**自动挂** `SkillsMiddleware` | 不用自己 `new`；它只解析 YAML front-matter，正文等模型来读 |

本节的改写（把示例指向本仓库的技能目录）：backend 的 `root_dir` 设成 `Agent/08_skills`，
`skills` 传 `["/skills/"]`，于是 `08_skills/skills/code-review-skill/SKILL.md` 会被发现。

课案那段 `image-to-mermaid` 的完整流程需要一张真实图片 + 一个调 qwen-vl 的脚本，
本机没有那份脚本，所以保留为「课案原文」讲解；真正跑起来的是上一课生成的
`code-review-skill`（纯文本技能，不需要外部视觉模型）。

### 1.1 路径常量、模型与课案原文

这一格只有三件事：三个路径常量（`skills` 参数要指向的那一层）、调度用的大模型、
以及原样抄下来的课案代码字符串（留着做对照）。

In [ ]:
# ============================================================
# 1.1 路径常量 + 模型 + 课案原文
# ============================================================
import json

from langchain.chat_models import init_chat_model

HERE = NB_DIR
# 技能的【父目录】：这个常量就是 skills= 参数要指向的那一层。
# 换成「具体技能目录」或「SKILL.md 文件本身」都会导致发现 0 个技能 ——
# 中间件是「ls 这一层 → 对每个子目录取里面的 SKILL.md」，它不认别的形状。
SKILLS_ROOT = HERE / "skills"          # 技能的【父目录】
SKILL_DIR = SKILLS_ROOT / "code-review-skill"

# 课案里用的是 ChatOpenAI(...)；本项目规范统一用 init_chat_model 指定 provider，
# 参数同样来自 settings，效果等价。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# ================================================================
# 课案原文（含被截断的注释），原样保留，便于对照
# ================================================================
COURSE_CODE = '''# 例子为把 图片转成mermaid skill
from langchain_openai import ChatOpenAI
from config import setting
from deepagents import create_deep_agent
from deepagents.backends.local_shell import LocalShellBackend

# 调度模型用文本大模型（工具调用能力强）；
# 视觉识别由 skill 里的脚本自己调 qwen-vl 完成，agent 本身不需要是 VL 模型
model = ChatOpenAI(model=setting.MODEL_NAME, api_key=setting.API_KEY, base_url=setting.BASE_URL)

agent = create_deep_agent(
    model=model,
    backend=LocalShellBackend(root_dir=".", inherit_env=True),
    skills=["/"],  # 扫描根目录 -> 发现 /image-to-mermaid/SKILL.md
)
result = agent.invoke({"messages": [{"role": "user", "content": (
    "请使用 image-to-mermaid 技能，把 img.png 转换成 Mermaid 架构图代码。"
    "注意：不要直接用 read_file 读取图片文件，而是按 SKILL.md 的说明运行其中的转换脚本。"
)}]})
print(result["messages"][-1].content)
'''

# 课案原文被截断的那行注释，补全后应该是这样（本节的第一个知识点）
COMPLETED_COMMENT = (
    "# 注意 skills 指向技能的【父目录】：\n"
    "#   skills=[\"/\"]                 → 扫描根目录下的所有一级子目录，发现 /image-to-mermaid/SKILL.md\n"
    "#   skills=[\"/skills/\"]          → 扫描 /skills/ 下的所有一级子目录\n"
    "#   skills=[\"/skills/a/\", \"/skills/b/\"] → 多个来源，后面的覆盖前面同名的技能\n"
    "# 不能写成某个技能目录（\"/image-to-mermaid/\"）或 SKILL.md 文件本身。"
)

print("常量与模型就绪：")
print("  SKILLS_ROOT =", SKILLS_ROOT)
print("  SKILL_DIR   =", SKILL_DIR)
print("  llm         =", settings.model_name)

### 预期输出

```text
常量与模型就绪：
  SKILLS_ROOT = F:\ProGram\Python_Base\Agent\08_skills\skills
  SKILL_DIR   = F:\ProGram\Python_Base\Agent\08_skills\skills\code-review-skill
  llm         = deepseek-flash
```

### 1.2 课案原文 + 被截断注释的补全

先把课案原文打印出来，再补全那行注释，最后补三点课案没展开、但写代码一定会踩的细节：

1. 路径是「backend 的虚拟路径」，不是宿主机路径。`root_dir="."` 时虚拟 `/` 就是当前
   工作目录；本节把 `root_dir` 设成 `Agent/08_skills`，于是虚拟 `/skills/` 落在
   `Agent/08_skills/skills/`。**换 root_dir 就要换 `skills` 里的路径。**
2. `LocalShellBackend` 的 `virtual_mode` 默认是 `True`，agent 只能用虚拟路径；
   配合 `inherit_env=True` 还会把本进程的环境变量传给子进程（技能里的脚本要用 API Key 时靠它）。
3. `skills` 参数一旦传了，`create_deep_agent` 会自动往中间件栈里塞一个 `SkillsMiddleware`
   （`deepagents/graph.py`），你不需要自己 `new` 它。

In [ ]:
# ============================================================
# 1.2 课案原文 + 被截断注释的补全
# ============================================================
def section_course_code() -> None:
    print("=" * 78)
    print("1. 课案原文（注意最后 `skills` 那行的注释是被截断的）")
    print("=" * 78)
    print(COURSE_CODE)
    print("-" * 78)
    print("课案原文里 `# 注意skills 指向技能的【父目` 这句被截断了，补全后是：")
    print()
    for line in COMPLETED_COMMENT.splitlines():
        print("    " + line)
    print()
    print(
        """
再补三点课案没展开、但写代码一定会踩的细节：

    (1) 路径是「backend 的虚拟路径」，不是宿主机路径。
        LocalShellBackend(root_dir=".") 时，虚拟 `/` = 当前工作目录；
        本文件把 root_dir 设成脚本所在目录 Agent/08_skills，
        于是虚拟 `/skills/` = Agent/08_skills/skills/。换 root_dir 就要换 skills 里的路径。

    (2) LocalShellBackend 的 virtual_mode 默认是 True，
        意味着 agent 只能用虚拟路径；配合 inherit_env=True 还会把本进程的
        环境变量传给子进程（技能里的脚本要用 API Key 时靠它）。

    (3) skills 参数一旦传了，create_deep_agent 会自动往中间件栈里塞一个
        SkillsMiddleware（见 deepagents/graph.py 第 863 行），
        你不需要自己 new 它。
"""
    )


section_course_code()

### 预期输出

```text
==============================================================================
1. 课案原文（注意最后 `skills` 那行的注释是被截断的）
==============================================================================
# 例子为把 图片转成mermaid skill
from langchain_openai import ChatOpenAI
from config import setting
from deepagents import create_deep_agent
from deepagents.backends.local_shell import LocalShellBackend

# 调度模型用文本大模型（工具调用能力强）；
# 视觉识别由 skill 里的脚本自己调 qwen-vl 完成，agent 本身不需要是 VL 模型
model = ChatOpenAI(model=setting.MODEL_NAME, api_key=setting.API_KEY, base_url=setting.BASE_URL)

agent = create_deep_agent(
    model=model,
    backend=LocalShellBackend(root_dir=".", inherit_env=True),
    skills=["/"],  # 扫描根目录 -> 发现 /image-to-mermaid/SKILL.md
)
result = agent.invoke({"messages": [{"role": "user", "content": (
    "请使用 image-to-mermaid 技能，把 img.png 转换成 Mermaid 架构图代码。"
    "注意：不要直接用 read_file 读取图片文件，而是按 SKILL.md 的说明运行其中的转换脚本。"
)}]})
print(result["messages"][-1].content)

------------------------------------------------------------------------------
课案原文里 `# 注意skills 指向技能的【父目` 这句被截断了，补全后是：

    # 注意 skills 指向技能的【父目录】：
    #   skills=["/"]                 → 扫描根目录下的所有一级子目录，发现 /image-to-mermaid/SKILL.md
    #   skills=["/skills/"]          → 扫描 /skills/ 下的所有一级子目录
    #   skills=["/skills/a/", "/skills/b/"] → 多个来源，后面的覆盖前面同名的技能
    # 不能写成某个技能目录（"/image-to-mermaid/"）或 SKILL.md 文件本身。


再补三点课案没展开、但写代码一定会踩的细节：

    (1) 路径是「backend 的虚拟路径」，不是宿主机路径。
        LocalShellBackend(root_dir=".") 时，虚拟 `/` = 当前工作目录；
        本文件把 root_dir 设成脚本所在目录 Agent/08_skills，
        于是虚拟 `/skills/` = Agent/08_skills/skills/。换 root_dir 就要换 skills 里的路径。

    (2) LocalShellBackend 的 virtual_mode 默认是 True，
        意味着 agent 只能用虚拟路径；配合 inherit_env=True 还会把本进程的
        环境变量传给子进程（技能里的脚本要用 API Key 时靠它）。

    (3) skills 参数一旦传了，create_deep_agent 会自动往中间件栈里塞一个
        SkillsMiddleware（见 deepagents/graph.py 第 863 行），
        你不需要自己 new 它。
```

### 1.3 技能发现预检：模型在实际对话前会拿到什么

这一格**不调模型**，做法是 `new` 一个 `SkillsMiddleware` 然后手动调 `before_agent` ——
这正是 `create_deep_agent` 内部会做的事，只是我们提前跑一遍看结果。

它同时充当**降级路径**：如果后面调模型失败（网络 / 额度 / 需要视觉模型），
至少还能看到「技能确实被发现了、系统提示词里长什么样」。

In [ ]:
# ============================================================
# 1.3 技能发现预检：看看模型到底会看到什么
# ============================================================
def section_discovery(backend, sources: list[str]) -> tuple[object, list[dict]]:
    """在不调大模型的前提下，先把技能发现结果打印出来。

    这一步同时充当**降级路径**：如果后面调模型失败（网络/额度/需要视觉模型），
    至少还能看到「技能确实被发现了、系统提示词里长什么样」。

    做法就是 new 一个 SkillsMiddleware 然后手动调 before_agent ——
    这正是 create_deep_agent 内部会做的事，只是我们提前跑一遍看结果。
    """
    print()
    print("=" * 78)
    print("2. 技能发现预检：模型在实际对话前会拿到什么")
    print("=" * 78)

    from deepagents.middleware.skills import SkillsMiddleware

    middleware = SkillsMiddleware(backend=backend, sources=sources)

    # before_agent(state, runtime, config) 只用到 state；传空 state 就会触发一次加载，
    # 因为它的逻辑是「state 里没有 skills_metadata 才加载」。
    update = middleware.before_agent({}, None, None)  # type: ignore[arg-type]
    skills_meta = list(update["skills_metadata"]) if update else []

    print(f"  技能来源（skills 参数）：{sources}")
    print(f"  来源标签（会渲染成系统提示词的小标题）：{middleware.source_labels}")
    print(f"  发现技能数：{len(skills_meta)}")
    print()
    print("  ── 以下文字会【原样】拼进 system prompt（这就是『选择』阶段的全部开销）──")
    print("  " + "-" * 74)
    for line in middleware._format_skills_locations().splitlines():  # noqa: SLF001
        print("  | " + line)
    print("  |")
    for line in middleware._format_skills_list(skills_meta).splitlines():  # noqa: SLF001
        print("  | " + line)
    print("  " + "-" * 74)
    print(
        """
  注意看：常驻系统提示词的只有 name + description + 路径，
  references/ 里那两份几十行的规范**一个字都没进去**。
  模型要读，得自己发起 read_file —— 这就是渐进式披露在工程上的落点。
"""
    )
    return middleware, skills_meta


# ---------- backend：与课案同构，root_dir 改成本仓库的技能父目录 ----------
if not (SKILL_DIR / "SKILL.md").exists():
    print()
    print("!" * 78)
    print(f"没有找到技能文件：{SKILL_DIR / 'SKILL.md'}")
    print("请先运行：uv run Agent/08_skills/02_SKILL示例_jxsd.py")
    print("!" * 78)
else:
    from deepagents.backends.local_shell import LocalShellBackend

    # root_dir：课案写的是 "."（当前工作目录）。这里改用脚本所在目录，
    #           好处是不管从哪个目录启动，虚拟路径 "/skills/" 都稳定落在
    #           Agent/08_skills/skills/。
    # inherit_env=True：技能里的脚本要用 .env 里的 API Key 时，靠它把环境变量传下去。
    backend = LocalShellBackend(
        root_dir=str(HERE),      # 课案是 "."；这里写死脚本目录，保证路径稳定
        inherit_env=True,        # 把本进程环境变量传给技能脚本（脚本要用 API Key 时靠它）
    )

    # 第 2 节：不需要模型的技能发现预检（同时也是降级输出）
    middleware, skills_meta = section_discovery(backend, ["/skills/"])

### 预期输出

```text

==============================================================================
2. 技能发现预检：模型在实际对话前会拿到什么
==============================================================================
  技能来源（skills 参数）：['/skills/']
  来源标签（会渲染成系统提示词的小标题）：['Skills']
  发现技能数：1

  ── 以下文字会【原样】拼进 system prompt（这就是『选择』阶段的全部开销）──
  --------------------------------------------------------------------------
  | **Skills Skills**: `/skills/` (higher priority)
  |
  | - **code-review-skill**: 审查 Python / JavaScript 代码，检查规范性、性能与安全问题。当用户要求「代码审查 / review / 检查代码 / 优化这段代码」时使用。 (License: MIT, Compatibility: 需要能读取项目源码目录；Python 3.10+)
  |   -> Allowed tools: read_file, write_file
  |   -> Read `/skills/code-review-skill/SKILL.md` for full instructions
  --------------------------------------------------------------------------

  注意看：常驻系统提示词的只有 name + description + 路径，
  references/ 里那两份几十行的规范**一个字都没进去**。
  模型要读，得自己发起 read_file —— 这就是渐进式披露在工程上的落点。
```

#### 实测：「`skills` 指向技能的父目录」到底有多严格

课案那行注释说得很肯定，但**没有给数据**。这一格用同一个 backend、只换 `sources`，
把四种写法各跑一遍 —— 这就是「写错一层就发现 0 个技能」的可复现证据。

In [ ]:
# ============================================================
# 1.3b 实测：sources 写错一层会怎样
# ============================================================
def section_pitfall(backend) -> None:
    """把课案注释里那句「skills 指向技能的【父目录】」跑成真数据。"""
    from deepagents.middleware.skills import SkillsMiddleware

    print()
    print("=" * 78)
    print("3. 实测：sources 写错一层会怎样（同一个 backend，只换 sources）")
    print("=" * 78)

    cases = [
        ("/skills/", "← 正确：技能的【父目录】"),
        ("/skills/code-review-skill/", "← 错：指向【具体技能目录】"),
        ("/skills/code-review-skill/SKILL.md", "← 错：直接指向 SKILL.md 文件"),
        ("/", "← 错：backend 根目录下没有直接摆技能目录"),
    ]
    for source, note in cases:
        probe = SkillsMiddleware(backend=backend, sources=[source])
        update = probe.before_agent({}, None, None)
        found = [s["name"] for s in (update["skills_metadata"] if update else [])]
        print(f"  sources=[{source!r}]".ljust(48) + f"发现 {len(found)} 个  {found}  {note}")

    print(
        """
  结论：skills 参数要指向【装着技能目录的那一层】。
        中间件的做法是「ls 这一层 → 对每个一级子目录找里面的 SKILL.md」，
        所以指向技能目录本身、或直接指向 SKILL.md 文件，都是 0 个。

        `skills=["/"]`（很多官方示例这么写）能成立，前提是技能目录就【直接摆在
        backend 根目录下】。本仓库的技能在 skills/ 下面一层，所以传 "/" 也是 0 个 ——
        这不是 bug，正是「必须指向父目录」这条规则的另一面。
"""
    )


if "backend" in dir():
    section_pitfall(backend)
else:
    print("（跳过 1.3b：技能目录缺失，backend 没有创建）")

### 预期输出

```text
Cannot load skills from '/skills/code-review-skill/SKILL.md': Path '/skills/code-review-skill/SKILL.md': not_a_directory
Skills load errors: ["Cannot load skills from '/skills/code-review-skill/SKILL.md': Path '/skills/code-review-skill/SKILL.md': not_a_directory"]

==============================================================================
3. 实测：sources 写错一层会怎样（同一个 backend，只换 sources）
==============================================================================
  sources=['/skills/']                          发现 1 个  ['code-review-skill']  ← 正确：技能的【父目录】
  sources=['/skills/code-review-skill/']        发现 0 个  []  ← 错：指向【具体技能目录】
  sources=['/skills/code-review-skill/SKILL.md']发现 0 个  []  ← 错：直接指向 SKILL.md 文件
  sources=['/']                                 发现 0 个  []  ← 错：backend 根目录下没有直接摆技能目录

  结论：skills 参数要指向【装着技能目录的那一层】。
        中间件的做法是「ls 这一层 → 对每个一级子目录找里面的 SKILL.md」，
        所以指向技能目录本身、或直接指向 SKILL.md 文件，都是 0 个。

        `skills=["/"]`（很多官方示例这么写）能成立，前提是技能目录就【直接摆在
        backend 根目录下】。本仓库的技能在 skills/ 下面一层，所以传 "/" 也是 0 个 ——
        这不是 bug，正是「必须指向父目录」这条规则的另一面。
```

### 1.4 真的创建 agent 并跑一次

参数与课案完全同构，只把 `root_dir` 换成 `NB_DIR`、`skills` 换成 `["/skills/"]`。
底下故意写了 5 处违反 `python_rules.md` 的代码（可变默认值、字符串拼价格、
未关闭文件、`eval`、缺少注解），让技能有活干。

关键看**工具调用轨迹**：模型有没有先读 `SKILL.md`、再按正文指示去读
`references/python_rules.md`。这就是「渐进式披露真的发生了」的证据。

> ⚠️ 模型轨迹每次都不同，下面的「预期输出」是**一次真实运行**的记录。

In [ ]:
# ============================================================
# 1.4 真的创建 agent 并跑一次
# ============================================================
# 故意写了 5 处违反 python_rules.md 的代码，让技能有活干
SAMPLE_CODE = '''def add_item(item, bucket=[]):
    """把 item 放进 bucket。"""
    bucket.append(item)
    return bucket


def total_price(orders):
    result = ""
    for o in orders:
        result += str(o["price"])
    return result


def load_config(name):
    f = open(name)
    data = f.read()
    return eval(data)
'''

USER_PROMPT = (
    "请使用 code-review-skill 技能审查下面这段 Python 代码。\n"
    "注意：不要凭印象评论，先按 SKILL.md 的说明去读对应的规范文件，再逐条比对给结论。\n\n"
    "```python\n" + SAMPLE_CODE + "```\n\n"
    "只输出审查结论，不要写任何文件。"
)


def section_run_agent(backend) -> None:
    print()
    print("=" * 78)
    print("3. 创建 agent 并真的跑一次（模型：" + settings.model_name + "）")
    print("=" * 78)

    from deepagents import create_deep_agent

    # ---------- 创建 agent（课案同一套参数）----------
    # root_dir：课案写的是 "."（当前工作目录）。这里改用脚本所在目录，
    #           好处是不管从哪个目录启动，虚拟路径 "/skills/" 都稳定落在
    #           Agent/08_skills/skills/。
    # inherit_env=True：技能里的脚本要用 .env 里的 API Key 时，靠它把环境变量传下去。
    agent = create_deep_agent(
        model=llm,
        backend=backend,
        skills=["/skills/"],  # ← 指向技能的【父目录】
        system_prompt="你是代码审查助手。优先使用可用的技能完成任务。",
    )

    print("  agent 创建成功，开始 invoke ...")
    print()

    result = agent.invoke(
        {"messages": [{"role": "user", "content": USER_PROMPT}]},
        config={"recursion_limit": 60},
    )

    # ---------- 打印工具调用轨迹：这是「渐进式披露真的发生了」的证据 ----------
    print("  ── 工具调用轨迹（看它有没有先读 SKILL.md、再读 python_rules.md）──")
    step = 0
    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "AIMessage":
            for call in getattr(message, "tool_calls", None) or []:
                step += 1
                args = json.dumps(call.get("args", {}), ensure_ascii=False)
                print(f"    [{step}] 模型 → 工具 {call.get('name')}  参数 {args[:150]}")
        elif kind == "ToolMessage":
            text = str(message.content).replace("\n", " ")
            print(f"        ↳ 工具返回（{getattr(message, 'name', '?')}）：{text[:110]} ...")
    if step == 0:
        print("    （本次没有工具调用 —— 模型直接作答了。可以重跑一次，或看下面的正文输出）")

    print()
    print("  ── 最终回答 ──")
    print("  " + "-" * 74)
    final = str(result["messages"][-1].content)
    # 完整回答可能有几千字（33 条规范逐条比对）；notebook 里截断到 900 字符，保证输出可读。
    if len(final) > 900:
        final = final[:900] + "\n…（答案已截断，完整回答见终端运行）"
    # 这一整格输出会被抄进 markdown 的代码块里；回答里如果自带反引号围栏，
    # 会把外面的块提前截断 —— 统一降级成单引号。
    final = final.replace("```", "'''")
    for line in final.splitlines():
        print("  | " + line)
    print("  " + "-" * 74)


# ================================================================
# 1.5 降级路径：调不动模型时，至少把技能发现结果和用法说清楚
# ================================================================
def section_degraded(reason: str) -> None:
    print()
    print("=" * 78)
    print("4. 降级演示（调用大模型这一步没有成功）")
    print("=" * 78)
    print(f"  原因：{reason}")
    print(
        """
  这不影响本节的核心结论 —— 技能【发现】完全在本地完成，不需要模型：

    1. create_deep_agent(skills=[...]) 会在启动时创建 SkillsMiddleware；
    2. 中间件 ls 技能父目录 → 对每个子目录取 SKILL.md → 只解析 YAML front-matter；
    3. 把 name / description / license / compatibility / allowed-tools
       拼成一小段文字塞进 system prompt；
    4. 任务匹配时，模型自己 read_file 拉全文 —— 这一步才需要模型。

  上面第 2 节的「技能发现预检」打印的就是第 3 步的结果，
  即使模型不可用，这份结果也是真实、可验证的。

  课案里 image-to-mermaid 那个技能还需要一张真实图片 + 一份调 qwen-vl 的脚本，
  本仓库没有那份脚本，所以那个例子只作为原文保留，不做实跑。
"""
    )


# 真跑。失败就走降级说明，绝不抛 traceback。
if "backend" not in dir():
    print("（跳过 1.4：上面前置自检里 local_skill 未就绪）")
elif not READY["model"]:
    print("（跳过 1.4：大模型未就绪，见前置条件自检）")
    section_degraded("settings.api_key / base_url / model_name 不完整")
else:
    try:
        section_run_agent(backend)
    except Exception as exc:  # noqa: BLE001 —— 教学脚本要保证「永远打印提示而不是 traceback」
        section_degraded(f"{type(exc).__name__}: {str(exc)[:200]}")

### 预期输出

> ⚠️ 本格输出含随机值 / 模型自由生成的内容，**每次运行会变**，下面只是其中一次真实运行的记录。

```text

==============================================================================
3. 创建 agent 并真的跑一次（模型：deepseek-flash）
==============================================================================
  agent 创建成功，开始 invoke ...

  ── 工具调用轨迹（看它有没有先读 SKILL.md、再读 python_rules.md）──
    [1] 模型 → 工具 read_file  参数 {"file_path": "/skills/code-review-skill/SKILL.md", "limit": 1000}
    [2] 模型 → 工具 ls  参数 {"path": "/skills/code-review-skill"}
        ↳ 工具返回（read_file）： 1  ---  2  name: code-review-skill  3  description: 审查 Python / JavaScript 代码，检查规范性、性能与安全问题。当用户要求「代码审查 / revi ...
        ↳ 工具返回（ls）：['/skills/code-review-skill/SKILL.md', '/skills/code-review-skill/references/'] ...
    [3] 模型 → 工具 ls  参数 {"path": "/skills/code-review-skill/references"}
    [4] 模型 → 工具 read_file  参数 {"file_path": "/skills/code-review-skill/references/python_rules.md", "limit": 1000}
        ↳ 工具返回（ls）：['/skills/code-review-skill/references/javascript_rules.md', '/skills/code-review-skill/references/python_rule ...
        ↳ 工具返回（read_file）： 1  # Python 代码审查规范（33 条）  2    3  > 本文件只在审查 Python 代码时才需要读入上下文 —— 这就是「渐进式披露」。  4    5  ## 一、命名与风格（PEP 8）  6   ...

  ── 最终回答 ──
  --------------------------------------------------------------------------
  | 已按 SKILL.md 流程执行：判定语言为 Python → 只读 `references/python_rules.md`（33 条），未读 JS 规范。以下逐条比对。
  | 
  | 行号按所给片段的自然顺序编号（片段未提供文件名）：L1 `def add_item`，L3 `bucket.append`，L8–L10 `result += ...`，L14 `def load_config`，L15 `open`，L17 `eval`。
  | 
  | ---
  | 
  | ## 一、逐条比对
  | 
  | ### 一、命名与风格
  | 
  | | 条 | 结论 | 证据 | 修法 |
  | |---|---|---|---|
  | | 1 模块名 | 不适用 | 片段无模块级信息 | — |
  | | 2 命名 snake_case | 通过 | `add_item` / `total_price` / `load_config` / `bucket` / `orders` 均合规 | — |
  | | 3 私有下划线 | 不适用 | 无私有成员 | — |
  | | 4 ≤100 字符、无反斜杠续行 | 通过 | 最长行 L17，远低于 100 | — |
  | | 5 import 分组 | 不适用 | 无 import | — |
  | | 6 禁止 `import *` | 不适用 | 同上 | — |
  | | 7 函数间空两行 | 通过 | L5–L6、L12–L13 各两个空行 | — |
  | | 8 注释写「为什么」 | **不通过** | L2 `"""把 item 放进 bucket。"""` —— 代码 `bucket.append(item)` 已经表达了这个「做了什么」 | 删除或改写为意图，如 `"""返回新 bucket（调用方传入的 bucket 不会被改动）。"""`（与第 32 条修法配套） |
  | 
  | ### 二、类型注解
  | 
  | | 条 | 结论 | 证据 | 修法 |
  | |---|---|---|---|
  | | 9 公开函数必须有参数/返回值注解 | **不通过** | L1 `def add_item(item, bucket=[])`、L7 `
  | …（答案已截断，完整回答见终端运行）
  --------------------------------------------------------------------------
```

## 2. 云技能：把 SKILL.md 存进 PostgreSQL，用 CompositeBackend 做多用户隔离

课案原文开头这段话就是本节的全部动机：

> DeepAgents 的 skills 机制原生支持两步加载：启动时只加载 SKILL.md 的 `description`
> （省 Token），Agent 匹配到技能后才 `read_file` 拉取完整提示词。
> 配合 `StoreBackend` 即可实现云端多用户隔离。

也就是说：

- 本地技能（第 1 节）放在磁盘上，**一台机器一份、改一次全体生效**；
- 云技能把 SKILL.md 存到 LangGraph 的 **Store（长期记忆）** 里，按 `namespace` 分区 ——
  `user-001` 和 `user-002` 各有各的技能库，互相看不见。

三个后端各自的位置（本节骨架）：

```python
CompositeBackend(
    default = StateBackend()      # 其它所有路径：走内存（会话内有效）
    routes  = {"/skills/": StoreBackend(namespace=...)}   # /skills/ 前缀 → 走 Store
)
```

于是模型看到的 `/skills/sql-gen/SKILL.md`，实际存在 PostgreSQL 的 `store` 表里，
`namespace = (用户,)`。

### ★ 与课案原文的两处【必要改动】（不改就跑不通 / 违反规范）★

| # | 课案原文 | 本 notebook | 为什么 |
|---|---|---|---|
| 1 | `DB = "postgresql://<用户>:<口令>@<主机>:<端口>/langgraph"` 硬编码 | `settings.pg_uri`（`.env`） | 规范铁律：口令 / 连接串不写进代码 |
| 2 | `store.put(ns, "/skills/code-review/SKILL.md", ...)` | `store.put(ns, "/code-review/SKILL.md", ...)` | `CompositeBackend` 命中路由 `/skills/` 后会把前缀**剥掉**再交给 `StoreBackend`，所以 key 要相对路由前缀。写全路径会让模型的虚拟路径变成 `/skills/skills/code-review/SKILL.md`，`skills=["/skills/"]` 一个技能都发现不了（实测：发现 0 个） |

另外本节在往 Store 写之前会先**清掉本 namespace 里的历史键**（见 2.4），
让「写入结果」那一行输出可复现 —— 否则课案脚本 / 历史运行留下的键会混进来。

### 2.1 课案原文、技能内容与命名空间

`create_file_data("""...""")` 的写法原样保留 —— 它把字符串包成 LangGraph Store
认识的文件结构（`content` / `encoding` / `created_at` / `modified_at`）。

> 注意 `name` 必须等于技能目录名（`code-review` / `sql-gen` / `weekly-report`），
> 否则中间件会告警。

In [ ]:
# ============================================================
# 2.1 课案原文 + 两个技能内容 + 命名空间
# ============================================================
import uuid

# ================================================================
# 课案原文（DB 那行做了脱敏，实际代码里必须用 settings.pg_uri）
# ================================================================
COURSE_CODE = '''from langchain.agents import create_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.store.postgres import PostgresStore
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_openai import ChatOpenAI
from config import settings

DB = "postgresql://<用户>:<口令>@<主机>:<端口>/langgraph"   # ← 课案是硬编码，已脱敏
model = ChatOpenAI(model=settings.model_name, api_key=settings.api_key, base_url=settings.base_url)

with (
    PostgresStore.from_conn_string(DB) as store,
    PostgresSaver.from_conn_string(DB) as checkpointer,
):
    store.setup()
    checkpointer.setup()

    # 为不同用户预存不同的 skill
    store.put(("user-001",), "/skills/code-review/SKILL.md", create_file_data("""---
name: code-review
description: Python 代码审查，检查规范性和性能问题
---


你是 Python 专家，审查以下代码的规范性、性能和安全性。
"""))

    store.put(("user-001",), "/skills/sql-gen/SKILL.md", create_file_data("""---
name: sql-gen
description: 根据自然语言生成 SQL 语句
---


根据用户的自然语言描述生成对应的 SQL 语句。
"""))

    agent = create_deep_agent(
        model=model,
        skills=["/skills/"],
        backend=CompositeBackend(
            default=StateBackend(),
            routes={
                "/skills/": StoreBackend(
                    namespace=lambda rt: ("user-001",),  # 本地用固定值
                ),
            },
        ),
        store=store,
        checkpointer=checkpointer,
    )
    config = {"configurable": {"thread_id": "1"}}
    result = agent.invoke({"messages": [{"role": "user", "content": "用技能sql-gen查询用户总数"}]}, config)
    for m in result["messages"]:
        print(m)
'''

# ================================================================
# 两个技能的内容：与课案一致，保留 create_file_data("""...""") 的写法
# ================================================================
# 注意 name 必须等于技能目录名（code-review / sql-gen），否则中间件会告警
CODE_REVIEW_SKILL = """---
name: code-review
description: Python 代码审查，检查规范性和性能问题
---


你是 Python 专家，审查以下代码的规范性、性能和安全性。
"""

SQL_GEN_SKILL = """---
name: sql-gen
description: 根据自然语言生成 SQL 语句
---


根据用户的自然语言描述生成对应的 SQL 语句。
"""

# 额外造一个「只有 user-002 有」的技能，用来演示多用户隔离
REPORT_SKILL = """---
name: weekly-report
description: 撰写正式工作周报时使用。当用户要求写周报、总结本周工作时使用。
---


按「本周完成 / 下周计划 / 风险与求助」三段写，总字数 200 字以内。
"""

USER_001 = "user-001"
USER_002 = "user-002"

print("三个技能文本就绪：code-review / sql-gen / weekly-report")
print("两个用户 namespace：", USER_001, "/", USER_002)

### 预期输出

```text
三个技能文本就绪：code-review / sql-gen / weekly-report
两个用户 namespace： user-001 / user-002
```

### 2.2 课案原文与两处必要改动

这一格把课案代码、以及本 notebook 相对它的改动逐条列出来（原因见本节开头的表格）。

In [ ]:
# ============================================================
# 2.2 课案原文与两处必要改动
# ============================================================
def section_course() -> None:
    print("=" * 78)
    print("1. 课案原文（连接串已脱敏；store key 见下方说明）")
    print("=" * 78)
    print(COURSE_CODE)
    print("-" * 78)
    # 用 ''' 做定界符：正文里要出现 create_file_data("""...""")，用 """ 会被提前截断
    print(
        f'''
本文件相对课案原文的改动：

    [改 1] 连接串
        课案：DB = "postgresql://<用户>:<口令>@<主机>:<端口>/langgraph"（硬编码，本文件已脱敏）
        本文件：settings.pg_uri
        原因：规范铁律 —— 口令/连接串不写进代码。当前 pg_uri 指向
              {settings.pg_uri.split('@')[-1] if '@' in settings.pg_uri else settings.pg_uri}
              （用户名口令部分不打印）

    [改 2] store.put 的 key 要去掉路由前缀
        课案：store.put(ns, "/skills/code-review/SKILL.md", ...)
        本文件：store.put(ns, "/code-review/SKILL.md", ...)
        原因：CompositeBackend 命中 "/skills/" 路由后会把前缀剥掉再交给
              StoreBackend（本机 deepagents 0.7.13 实测）。
              用课案的写法，模型虚拟路径会变成 /skills/skills/code-review/SKILL.md，
              而 skills=["/skills/"] 会发现 0 个技能 —— 本文件在注释里保留了
              这个坑的说明，代码走正确写法。

    [保持] create_file_data("""...""") 的写法原样保留 ——
           它把字符串包成 LangGraph Store 认识的文件结构
           （content / encoding / created_at / modified_at）。
'''
    )


section_course()

### 预期输出

```text
==============================================================================
1. 课案原文（连接串已脱敏；store key 见下方说明）
==============================================================================
from langchain.agents import create_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.backends.utils import create_file_data
from langgraph.store.postgres import PostgresStore
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_openai import ChatOpenAI
from config import settings

DB = "postgresql://<用户>:<口令>@<主机>:<端口>/langgraph"   # ← 课案是硬编码，已脱敏
model = ChatOpenAI(model=settings.model_name, api_key=settings.api_key, base_url=settings.base_url)

with (
    PostgresStore.from_conn_string(DB) as store,
    PostgresSaver.from_conn_string(DB) as checkpointer,
):
    store.setup()
    checkpointer.setup()

    # 为不同用户预存不同的 skill
    store.put(("user-001",), "/skills/code-review/SKILL.md", create_file_data("""---
name: code-review
description: Python 代码审查，检查规范性和性能问题
---


你是 Python 专家，审查以下代码的规范性、性能和安全性。
"""))

    store.put(("user-001",), "/skills/sql-gen/SKILL.md", create_file_data("""---
name: sql-gen
description: 根据自然语言生成 SQL 语句
---


根据用户的自然语言描述生成对应的 SQL 语句。
"""))

    agent = create_deep_agent(
        model=model,
        skills=["/skills/"],
        backend=CompositeBackend(
            default=StateBackend(),
            routes={
                "/skills/": StoreBackend(
                    namespace=lambda rt: ("user-001",),  # 本地用固定值
                ),
            },
        ),
        store=store,
        checkpointer=checkpointer,
    )
    config = {"configurable": {"thread_id": "1"}}
    result = agent.invoke({"messages": [{"role": "user", "content": "用技能sql-gen查询用户总数"}]}, config)
    for m in result["messages"]:
        print(m)

------------------------------------------------------------------------------

本文件相对课案原文的改动：

    [改 1] 连接串
        课案：DB = "postgresql://<用户>:<口令>@<主机>:<端口>/langgraph"（硬编码，本文件已脱敏）
        本文件：settings.pg_uri
        原因：规范铁律 —— 口令/连接串不写进代码。当前 pg_uri 指向
              127.0.0.1:5432/langgraph
              （用户名口令部分不打印）

    [改 2] store.put 的 key 要去掉路由前缀
        课案：store.put(ns, "/skills/code-review/SKILL.md", ...)
        本文件：store.put(ns, "/code-review/SKILL.md", ...)
        原因：CompositeBackend 命中 "/skills/" 路由后会把前缀剥掉再交给
              StoreBackend（本机 deepagents 0.7.13 实测）。
              用课案的写法，模型虚拟路径会变成 /skills/skills/code-review/SKILL.md，
              而 skills=["/skills/"] 会发现 0 个技能 —— 本文件在注释里保留了
              这个坑的说明，代码走正确写法。

    [保持] create_file_data("""...""") 的写法原样保留 ——
           它把字符串包成 LangGraph Store 认识的文件结构
           （content / encoding / created_at / modified_at）。
```

### 2.3 连接 PostgreSQL（Store + Checkpointer）

源文件用**一个 `with` 同时开两个连接上下文**：

```python
with (
    PostgresStore.from_conn_string(DB) as store,
    PostgresSaver.from_conn_string(DB) as checkpointer,
):
    store.setup()
    checkpointer.setup()
```

notebook 里后面几格要**跨格共用**这两个对象，所以改用 `contextlib.ExitStack`
一次开好、最后一格统一 `close()` —— 语义等价，只是把 `with` 的生命周期摊到了多格。

| 对象 | 管什么 | 分区键 |
|---|---|---|
| `PostgresStore`（store） | 跨会话的**长期数据** | `namespace` —— 本节的技能文件就放这一层 |
| `PostgresSaver`（checkpointer） | **会话消息**检查点 | `thread_id` —— 多轮对话能续上 |

两者可以共用一个库（本机都在 `langgraph` 库里，表名不同）。`setup()` 必须各调一次：
自动建表，幂等，重复调用没问题。

In [ ]:
# ============================================================
# 2.3 连接 PostgreSQL + 建表（幂等）
# ============================================================
CLOUD_READY = False
_stack = None

if not READY["pg"]:
    print("！.env 里没有配置 PG_URI，请在 F:\\ProGram\\Python_Base\\.env 中补上，")
    print("  形如：PG_URI=postgresql://<用户>:<口令>@127.0.0.1:5432/langgraph")
    print("（跳过整个第 2 节）")
else:
    try:
        # 延迟 import：把「缺包」也纳入可读提示的范围
        from langgraph.checkpoint.postgres import PostgresSaver
        from langgraph.store.postgres import PostgresStore

        # 源文件是 `with (PostgresStore... as store, PostgresSaver... as checkpointer):`
        # 一个 with 同时开两个连接上下文；notebook 要跨多格共用 → 换 ExitStack。
        from contextlib import ExitStack

        _stack = ExitStack()
        store = _stack.enter_context(PostgresStore.from_conn_string(settings.pg_uri))
        checkpointer = _stack.enter_context(PostgresSaver.from_conn_string(settings.pg_uri))

        # setup() 必须各调一次：自动建表（幂等，重复调用没问题）
        store.setup()
        checkpointer.setup()

        CLOUD_READY = True
        print("PostgreSQL 已连接 + 两张表已就绪")
        print("  库名（连接串尾部）：", settings.pg_uri.split("/")[-1])
        print("  store        ← 长期记忆：技能文件按 namespace 分区")
        print("  checkpointer ← 会话检查点：按 thread_id 分区")
    except ImportError as exc:
        print()
        print(f"！缺少依赖：{exc}")
        print("  请在项目根目录执行：uv add langgraph-checkpoint-postgres langgraph-checkpoint")
    except Exception as exc:  # noqa: BLE001 —— 教学脚本不允许抛 traceback
        print()
        print("！连接 PostgreSQL 失败：", f"{type(exc).__name__}: {str(exc)[:200]}")
        print("  排查顺序：")
        print("    1. PostgreSQL 是否已启动（默认 127.0.0.1:5432）")
        print("    2. langgraph 库是否已建：CREATE DATABASE langgraph;")
        print("    3. .env 里的 PG_URI 用户名/口令/库名是否正确")

### 预期输出

```text
PostgreSQL 已连接 + 两张表已就绪
  库名（连接串尾部）： langgraph
  store        ← 长期记忆：技能文件按 namespace 分区
  checkpointer ← 会话检查点：按 thread_id 分区
```

### 2.4 把技能写进 Store（按 namespace 隔离）

`store.put(namespace, key, value)` 的两个参数是本节的重点：

- `namespace`：分区键，相当于「谁的技能库」。这里用固定的用户标识；
  真实项目里从鉴权信息里取（课案注释：「本地用固定值」），就是一用户一技能库。
- `key`：文件在 `StoreBackend` 里的路径 —— **相对于 `CompositeBackend` 的路由前缀**。

In [ ]:
# ============================================================
# 2.4 把技能写进 PostgreSQL 的 Store（按 namespace 隔离）
# ============================================================
def section_seed_store(store) -> None:
    print("=" * 78)
    print("2. 把技能写进 PostgreSQL 的 Store（按 namespace 隔离）")
    print("=" * 78)

    from deepagents.backends.utils import create_file_data

    # ★ notebook 专属改动（理由见「与源文件的差异」）：
    #   先把本 namespace 里的历史键清掉 —— Store 是覆盖写，不清也不报错，
    #   但课案脚本 / 早期运行留下的键会一起出现在下面的「写入结果」里，
    #   预期输出就不可复现了。教学 notebook 要能反复跑出同一份结果。
    for user in (USER_001, USER_002):
        for item in store.search((user,)):
            store.delete((user,), item.key)

    # store.put(namespace, key, value)
    #   namespace：分区键，相当于「谁的技能库」；这里用 ("user-001",) 固定值，
    #              真实项目里可以从鉴权信息里取，实现一用户一技能库。
    #   key      ：文件在 StoreBackend 里的路径 —— 相对于 CompositeBackend 的路由前缀。
    store.put((USER_001,), "/code-review/SKILL.md", create_file_data(CODE_REVIEW_SKILL))
    print(f"  [{USER_001}] + /code-review/SKILL.md")
    store.put((USER_001,), "/sql-gen/SKILL.md", create_file_data(SQL_GEN_SKILL))
    print(f"  [{USER_001}] + /sql-gen/SKILL.md")

    # user-002 只有一个技能 —— 用来证明两个用户的技能库互相看不见
    store.put((USER_002,), "/weekly-report/SKILL.md", create_file_data(REPORT_SKILL))
    print(f"  [{USER_002}] + /weekly-report/SKILL.md")

    print()
    print("  写入结果（直接读 Store，验证真的落库了）：")
    for user in (USER_001, USER_002):
        items = store.search((user,))
        print(f"    namespace=({user!r},) → {[item.key for item in items]}")


if CLOUD_READY:
    section_seed_store(store)
else:
    print("（跳过 2.4：PostgreSQL 未就绪，排查顺序见上一格）")

### 预期输出

```text
==============================================================================
2. 把技能写进 PostgreSQL 的 Store（按 namespace 隔离）
==============================================================================
  [user-001] + /code-review/SKILL.md
  [user-001] + /sql-gen/SKILL.md
  [user-002] + /weekly-report/SKILL.md

  写入结果（直接读 Store，验证真的落库了）：
    namespace=('user-001',) → ['/sql-gen/SKILL.md', '/code-review/SKILL.md']
    namespace=('user-002',) → ['/weekly-report/SKILL.md']
```

### 2.5 用 CompositeBackend + StoreBackend 跑 agent

`build_agent` 就是「同一套代码，只换 namespace」：

- `default=StateBackend()`：非 `/skills/` 的路径留在内存里（会话结束即失效）；
- `routes={"/skills/": StoreBackend(...)}`：命中前缀的路径交给 Store，落到 PostgreSQL；
- `namespace=lambda rt: (user_id,)`：**运行时 → 命名空间**的函数。
  课案注释写的是「本地用固定值」；真实项目里改成读鉴权信息就能按登录用户隔离。

`thread_id` 用 `uuid`：每次运行都是全新会话。课案写的是固定 `"1"`，但那样第二次运行
会命中 PostgreSQL 里上一轮的检查点，而 `SkillsMiddleware` 的逻辑是
「state 里已有 `skills_metadata` 就不再加载」，结果重跑时技能加载被跳过、行为不稳定。
教学脚本要可重复运行，故用 uuid。

> ⚠️ 这一格的输出里既有随机 uuid，又有模型自己生成的回答 —— **每次运行会变**，
> 下面的「预期输出」只是其中一次真实运行的记录。

In [ ]:
# ============================================================
# 2.5 用 CompositeBackend + StoreBackend 跑 agent
# ============================================================
def build_agent(store, checkpointer, user_id: str):
    """按用户组装 agent：同一个 model / 同一套代码，只有 namespace 不同。"""
    from deepagents import create_deep_agent
    from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

    return create_deep_agent(
        model=llm,
        skills=["/skills/"],  # 技能的【父目录】——云技能同样遵守这条规则
        backend=CompositeBackend(
            default=StateBackend(),  # 非 /skills/ 的路径留在内存里（会话结束即失效）
            routes={
                # 命中 "/skills/" 的路径全部交给 StoreBackend，落到 PostgreSQL
                "/skills/": StoreBackend(
                    # namespace 是「运行时 → 命名空间」的函数。
                    # 课案注释写的是「本地用固定值」；真实项目里改成
                    # lambda rt: (rt.context.user_id,) 之类，就能按登录用户隔离。
                    namespace=lambda rt: (user_id,),
                    store=store,
                ),
            },
        ),
        store=store,                 # 让 agent 的长期记忆工具也用这个 store
        checkpointer=checkpointer,   # 会话检查点：多轮对话能续上
    )


def section_run(store, checkpointer) -> None:
    print()
    print("=" * 78)
    print("3. 真跑一次：让 agent 用云端的 sql-gen 技能")
    print("=" * 78)

    agent = build_agent(store, checkpointer, USER_001)

    # thread_id 用 uuid：每次运行都是全新会话。
    # 课案写的是固定 "1"，但那样第二次运行会命中 PostgreSQL 里上一轮的检查点，
    # 而 SkillsMiddleware 的逻辑是「state 里已有 skills_metadata 就不再加载」，
    # 结果就是重跑时技能加载被跳过、行为不稳定。教学脚本要可重复运行，故用 uuid。
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    print(f"  thread_id = {config['configurable']['thread_id']}")
    print("  用户输入：用技能 sql-gen 查询用户总数")
    print()

    result = agent.invoke(
        {"messages": [{"role": "user", "content": "用技能sql-gen查询用户总数"}]},
        config,
    )

    # 只打印最后一条 AI 回答 + 工具调用轨迹（课案是 for m in result["messages"]: print(m)，
    # 那样会把每条消息对象全量打出来，几十行噪音；这里保留等价信息但可读）
    print("  ── 工具调用轨迹 ──")
    for message in result["messages"]:
        if type(message).__name__ == "AIMessage":
            for call in getattr(message, "tool_calls", None) or []:
                print(f"    模型 → {call.get('name')}  参数 {call.get('args')}")
    print()
    print("  ── 最终回答 ──")
    print("  " + "-" * 74)
    final = str(result["messages"][-1].content)
    # 输出会被抄进 markdown 的代码块里，把回答自带的反引号围栏降级，避免提前截断。
    final = final.replace("```", "'''")
    for line in final.splitlines():
        print("  | " + line)
    print("  " + "-" * 74)


if CLOUD_READY:
    if not READY["model"]:
        print("（跳过 2.5：大模型未就绪，见前置条件自检）")
    else:
        try:
            section_run(store, checkpointer)
        except Exception as exc:  # noqa: BLE001
            print()
            print("！调用模型失败：", f"{type(exc).__name__}: {str(exc)[:200]}")
            print("  技能已经写进 Store（上一格），可以先看 2.6 的隔离验证。")
else:
    print("（跳过 2.5：PostgreSQL 未就绪）")

### 预期输出

> ⚠️ 本格输出含随机值 / 模型自由生成的内容，**每次运行会变**，下面只是其中一次真实运行的记录。

```text

==============================================================================
3. 真跑一次：让 agent 用云端的 sql-gen 技能
==============================================================================
  thread_id = 563d7981-4535-4fbe-a56e-eaaed142331c
  用户输入：用技能 sql-gen 查询用户总数

  ── 工具调用轨迹 ──
    模型 → read_file  参数 {'file_path': '/skills/sql-gen/SKILL.md', 'limit': 1000}
    模型 → ls  参数 {'path': '/skills/sql-gen'}
    模型 → ls  参数 {'path': '/'}

  ── 最终回答 ──
  --------------------------------------------------------------------------
  | 我读取了 `/skills/sql-gen/SKILL.md`，技能内容很简短，只说明「根据自然语言描述生成 SQL 语句」，且该目录下没有其他配置文件（无表结构定义、无数据库连接信息、无可执行脚本）。
  | 
  | **生成的 SQL：**
  | 
  | '''sql
  | SELECT COUNT(*) AS user_count
  | FROM users;
  | '''
  | 
  | **说明与假设：**
  | 
  | | 项 | 内容 |
  | |---|---|
  | | 表名 | 假设为 `users`（如实际为 `t_user`、`sys_user` 等请告知） |
  | | 统计口径 | 统计表内全部行数 |
  | 
  | **可能需要按实际情况调整：**
  | 
  | - 若表使用了**逻辑删除**，通常需要排除已删除记录：
  | 
  |   '''sql
  |   SELECT COUNT(*) AS user_count
  |   FROM users
  |   WHERE is_deleted = 0;   -- 或 deleted_at IS NULL
  |   '''
  | 
  | - 若只想统计**有效/激活**用户：
  | 
  |   '''sql
  |   SELECT COUNT(*) AS user_count
  |   FROM users
  |   WHERE status = 1;
  |   '''
  | 
  | 两点补充说明:
  | 
  | 1. **无法实际执行查询** — 当前环境里没有数据库连接，也没有任何可用的 SQL 执行工具（文件系统下只有 `/skills/`），所以上面是生成的语句而非查询结果。如果你能提供数据库连接方式或表结构文件，我可以帮你进一步走通执行环节。
  | 2. 该技能的 SKILL.md 内容偏薄，缺少表结构/方言（MySQL、PostgreSQL 等）约定，因此这里给的是通用标准 SQL。如果你告诉我具体的表和数据库类型，我可以生成更精确的版本。
  --------------------------------------------------------------------------
```

### 2.6 多用户隔离验证：同一份代码，只换 namespace

两个用户各建一套 backend + middleware，**唯一的差别就是 namespace**。

这一格还顺手演示了一个经典坑：**循环里建 lambda**。这里每个 `user_id` 都在循环内
立刻用掉（`before_agent` 马上执行），所以直接闭包捕获 `user_id` 是安全的；
但如果把这些对象先存起来、出了循环再统一调用，闭包就会全部读到最后一个 `user_id` ——
写成 `lambda rt, u=user_id: ...`（默认参数）才保险。

In [ ]:
# ============================================================
# 2.6 多用户隔离验证：同一份代码，只换 namespace
# ============================================================
def section_isolation(store, checkpointer) -> None:
    print()
    print("=" * 78)
    print("4. 多用户隔离验证：同一份代码，只换 namespace")
    print("=" * 78)

    from deepagents.middleware.skills import SkillsMiddleware

    from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

    # 两个用户各建一套 backend + middleware，唯一的差别就是 namespace。
    # 这里每个 user_id 都在循环内【立刻】用掉（before_agent 马上执行），
    # 所以下面的 lambda 直接闭包捕获 user_id 是安全的；如果把这些对象先存起来、
    # 出了循环再统一调用，闭包就会全部读到最后一个 user_id —— 「循环里建 lambda」
    # 的经典坑，写成 lambda rt, u=user_id: (u,) 才保险。
    for user_id in (USER_001, USER_002):
        backend = CompositeBackend(
            default=StateBackend(),
            routes={"/skills/": StoreBackend(namespace=lambda rt, u=user_id: (u,), store=store)},
        )
        middleware = SkillsMiddleware(backend=backend, sources=["/skills/"])
        update = middleware.before_agent({}, None, None)  # type: ignore[arg-type]
        names = [s["name"] for s in (update["skills_metadata"] if update else [])]
        print(f"  namespace=({user_id!r},) 发现的技能 → {names}")

    print(
        """
  结论：两份数据存在同一张 PostgreSQL 表里，靠 namespace 分区；
        user-002 无论怎么问，都不可能看到 user-001 的 code-review / sql-gen。
        这就是课案说的「配合 StoreBackend 即可实现云端多用户隔离」——
        隔离的粒度不是机器，也不是目录，而是 store 的 namespace。

  顺带说明另外两层的分工：
        PostgresSaver（checkpointer）存的是【会话消息】，
        按 thread_id 分区，负责「多轮对话记得住」；
        PostgresStore（store）存的是【跨会话的长期数据】，
        按 namespace 分区，本节的技能文件就放在这一层。
        两者可以共用一个库（本机都在 langgraph 库里，表名不同）。
"""
    )


if CLOUD_READY:
    section_isolation(store, checkpointer)
else:
    print("（跳过 2.6：PostgreSQL 未就绪）")

### 预期输出

```text

==============================================================================
4. 多用户隔离验证：同一份代码，只换 namespace
==============================================================================
  namespace=('user-001',) 发现的技能 → ['code-review', 'sql-gen']
  namespace=('user-002',) 发现的技能 → ['weekly-report']

  结论：两份数据存在同一张 PostgreSQL 表里，靠 namespace 分区；
        user-002 无论怎么问，都不可能看到 user-001 的 code-review / sql-gen。
        这就是课案说的「配合 StoreBackend 即可实现云端多用户隔离」——
        隔离的粒度不是机器，也不是目录，而是 store 的 namespace。

  顺带说明另外两层的分工：
        PostgresSaver（checkpointer）存的是【会话消息】，
        按 thread_id 分区，负责「多轮对话记得住」；
        PostgresStore（store）存的是【跨会话的长期数据】，
        按 namespace 分区，本节的技能文件就放在这一层。
        两者可以共用一个库（本机都在 langgraph 库里，表名不同）。
```

### 2.7 收尾：关掉连接

对应源文件里 `with (...)` 块退出时的行为。notebook 是长驻进程，
不主动关会一直占着连接池 —— 所以最后一格统一收尾。

In [ ]:
# ============================================================
# 2.7 收尾：关掉 store / checkpointer 的连接池
# ============================================================
if _stack is not None:
    _stack.close()
    print("PostgreSQL 连接已关闭（对应源文件 with 块退出时的行为）")
else:
    print("（本次没有建立 PostgreSQL 连接，无需关闭）")

### 预期输出

```text
PostgreSQL 连接已关闭（对应源文件 with 块退出时的行为）
```

## 3. Claude Code 技能：装在哪、怎么装

这一节讲「技能写完之后，怎么装到真实客户端里」。核心是三条：

1. **加载路径**（课案按优先级排序）：全局 `~/.claude/skills/`（本机所有项目生效）、
   项目 `<项目目录>/.claude/skills/`（只有这个项目生效）。
   复制一个技能目录到这两个目录任何一个，**就完成安装了**。
2. **自动创建技能**四步：① 下载 `ComposioHQ/awesome-claude-skills`；
   ② 把其中的 `skill-creator` 放进 `~/.claude/skills/`；③ 打开 PowerShell 输入 `Claude`；
   ④ 说「帮我创建一个 skill，使用 qwen3-vl 将流程图的图片转 mermaid」。
3. **各平台支持**：Claude Code、Cursor、Trae / OpenCode 均支持免费使用 Skills，
   其中 Claude Code 为官方推荐、功能最丰富。

本节不只打印文字 —— 它会**真的把上一课生成的技能复制一份**到项目级安装路径。

### ★ 关于优先级的实测提醒 ★

课案把全局排在①、项目排在②。而本机 `deepagents` 0.7.13 的 `SkillsMiddleware`
的规则是「后面的来源覆盖前面同名的技能（last one wins）」，并在系统提示词里
把最后一个来源标成 `(higher priority)`。所以用代码挂多个来源时，要把
【更具体的那个（项目级）放在后面】：

```python
skills=["/skills/", "/.claude/skills/"]
        ↑ 基础/全局            ↑ 项目级，优先级更高
```

两者说法不矛盾 —— 课案讲的是「客户端先看哪个目录」，框架讲的是「数组里谁在后面」，
接线时按框架的规则写就不会错。3.5 会把这条规则跑出来。

### 3.1 四个路径常量与目录树打印

本节全部动作都围着这几个常量转：

| 常量 | 是什么 |
|---|---|
| `SKILLS_ROOT` | 上一课生成技能的地方，也是「技能的父目录」 |
| `SOURCE_SKILL` | 要安装的那个技能（一个目录，含 `SKILL.md` + `references/`） |
| `PROJECT_CLAUDE_DIR` / `INSTALLED_SKILL` | 项目级安装**目的地** |
| `GLOBAL_CLAUDE_SKILLS` | 全局路径；本节只对它做**只读列举**，绝不写入 |

目录树按**行数**显示而不是字节数：技能文件是 markdown，行数更有意义；
二进制 / 非 UTF-8 文件（图片、`.pyc`）读不了，退化成显示字节数。

In [ ]:
# ============================================================
# 3.1 路径常量 + 目录树打印
# ============================================================
import shutil

HERE = NB_DIR
# 四个路径常量，本节全部动作都围着它们转：
#   SKILLS_ROOT   = 02 小节生成技能的地方，也是「技能的父目录」
#   SOURCE_SKILL  = 要安装的那个技能（一个目录，含 SKILL.md + references/）
#   PROJECT_CLAUDE_DIR / INSTALLED_SKILL = 项目级安装【目的地】
#   GLOBAL_CLAUDE_SKILLS = 全局路径；本节只对它做【只读列举】，绝不写入
SKILLS_ROOT = HERE / "skills"                 # 技能的父目录（02 小节生成）
SOURCE_SKILL = SKILLS_ROOT / "code-review-skill"
PROJECT_CLAUDE_DIR = HERE / ".claude" / "skills"   # 项目级安装路径
INSTALLED_SKILL = PROJECT_CLAUDE_DIR / "code-review-skill"
# expanduser("~") 展开成 C:\Users\<用户名>；用 Path 拼接而不是裸字符串拼，
# 这样反斜杠/正斜杠的差异交给 pathlib 处理，脚本在 Windows 与 Linux 上都能跑。
GLOBAL_CLAUDE_SKILLS = Path(os.path.expanduser("~")) / ".claude" / "skills"


# ================================================================
# 目录树打印（安装前后各一次，用来对比）
# ================================================================
def print_tree(root: Path, title: str) -> None:
    print(f"  【{title}】{root}")
    if not root.exists():
        print("    （目录不存在）")
        return

    def walk(path: Path, prefix: str = "") -> None:
        children = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
        for index, child in enumerate(children):
            # 跳过 __pycache__：里面的 .pyc 是二进制，读不出「行数」
            if child.is_dir() and child.name == "__pycache__":
                continue
            is_last = index == len(children) - 1
            branch = "└── " if is_last else "├── "
            if child.is_dir():
                print(f"    {prefix}{branch}{child.name}/")
                walk(child, prefix + ("    " if is_last else "│   "))
            else:
                # 显示行数而不是字节数：技能文件是 markdown，行数更有意义。
                # 二进制/非 UTF-8 文件（图片、.pyc）读不了，退化成显示字节数。
                try:
                    lines = len(child.read_text(encoding="utf-8").splitlines())
                    note = f"{lines} 行"
                except (UnicodeDecodeError, OSError):
                    note = f"{child.stat().st_size} 字节（二进制）"
                print(f"    {prefix}{branch}{child.name}    # {note}")

    walk(root)


print("路径常量：")
print("  SOURCE_SKILL        =", SOURCE_SKILL)
print("  PROJECT_CLAUDE_DIR  =", PROJECT_CLAUDE_DIR)
print("  GLOBAL_CLAUDE_SKILLS=", GLOBAL_CLAUDE_SKILLS, "（只读）")

### 预期输出

```text
路径常量：
  SOURCE_SKILL        = F:\ProGram\Python_Base\Agent\08_skills\skills\code-review-skill
  PROJECT_CLAUDE_DIR  = F:\ProGram\Python_Base\Agent\08_skills\.claude\skills
  GLOBAL_CLAUDE_SKILLS= C:\Users\QianXi\.claude\skills （只读）
```

### 3.2 加载路径与优先级

两个位置怎么选：

| 位置 | 生效范围 | 适合放什么 |
|---|---|---|
| `~/.claude/skills/` | 本机所有项目 | 通用技能：文档处理、代码审查、爬虫 |
| `<项目>/.claude/skills/` | 只有这个项目 | 项目专属：本仓库的接口规范、数据字典 |

安装动作本质上就是「把一个技能目录复制过去」，没有任何注册步骤：

```powershell
# PowerShell（本机 shell）
Copy-Item -Recurse -Force <技能目录> $env:USERPROFILE\.claude\skills\
# bash（远程 Linux 服务器）
cp -r <技能目录> ~/.claude/skills/
```

全局目录这一格只做**只读列举**。顺带认一下 Windows 特有的「目录联接（junction）」：
`is_symlink()` 对它返回 `False`，要用 `os.path.isjunction()` 才认得出。

In [ ]:
# ============================================================
# 3.2 Claude Code 的加载路径与优先级
# ============================================================
def section_load_paths() -> None:
    print("=" * 78)
    print("1. Claude Code Skills 的加载路径（按优先级排序）")
    print("=" * 78)
    print(
        f"""
课案原文：
    1. 全局：`~/.claude/skills/`（全局生效）
    2. 项目：项目目录下的 `.claude/skills/`（项目专用）
    复制 image-to-mermaid 到以上两个目录任何一个，就完成安装了。

放到本机上，两个路径分别是：

    全局：{GLOBAL_CLAUDE_SKILLS}
    项目：{PROJECT_CLAUDE_DIR}
          （也就是本文件所在的 Agent/08_skills/ 下面）

两个位置怎么选：

    | 位置 | 生效范围 | 适合放什么 |
    |---|---|---|
    | ~/.claude/skills/      | 本机所有项目 | 通用技能：文档处理、代码审查、爬虫… |
    | <项目>/.claude/skills/ | 只有这个项目 | 项目专属：本仓库的接口规范、数据字典… |

安装动作本质上就是「把一个技能目录复制过去」，没有任何注册步骤：

    # PowerShell（本机 shell）
    Copy-Item -Recurse -Force <技能目录> $env:USERPROFILE\\.claude\\skills\\
    # bash（远程 Linux 服务器）
    cp -r <技能目录> ~/.claude/skills/
"""
    )

    # 只读列举全局目录 —— 本节不往全局目录写任何东西
    print("  本机全局路径的当前状态（只读列举，不做修改）：")
    if GLOBAL_CLAUDE_SKILLS.exists():
        # Windows 上还有一种「目录联接（junction）」：is_symlink() 返回 False，
        # 需要用 os.path.isjunction() 才认得出。本机 ~/.claude/skills 就是联接。
        is_junction = getattr(os.path, "isjunction", lambda _p: False)(GLOBAL_CLAUDE_SKILLS)
        if GLOBAL_CLAUDE_SKILLS.is_symlink():
            kind = "符号链接"
        elif is_junction:
            kind = "目录联接（junction）"
        else:
            kind = "真实目录"
        target = ""
        if GLOBAL_CLAUDE_SKILLS.is_symlink() or is_junction:
            try:
                target = f"  → {os.readlink(GLOBAL_CLAUDE_SKILLS)}"
            except OSError:
                target = "  → （读取链接目标失败）"
        names = sorted(p.name for p in GLOBAL_CLAUDE_SKILLS.iterdir() if p.is_dir())
        print(f"    类型：{kind}{target}")
        print(f"    已安装全局技能 {len(names)} 个：{names}")
    else:
        print("    （本机还没有 ~/.claude/skills/ 目录；要装全局技能就先建它）")

    print(
        """
  ★ 实测提醒：多个来源时的覆盖方向
      本机 deepagents 0.7.13 的 SkillsMiddleware 规则是「后面的来源覆盖前面同名的
      技能」（last one wins），并在系统提示词里把【最后一个】来源标成
      (higher priority)。所以代码里挂多个来源时，把项目级放【后面】：

          skills=["/skills/", "/.claude/skills/"]
                  ↑ 基础/全局            ↑ 项目级，优先级更高

      本节最后的验证环节会把这条规则跑出来给你看。
"""
    )


section_load_paths()

### 预期输出

```text
==============================================================================
1. Claude Code Skills 的加载路径（按优先级排序）
==============================================================================

课案原文：
    1. 全局：`~/.claude/skills/`（全局生效）
    2. 项目：项目目录下的 `.claude/skills/`（项目专用）
    复制 image-to-mermaid 到以上两个目录任何一个，就完成安装了。

放到本机上，两个路径分别是：

    全局：C:\Users\QianXi\.claude\skills
    项目：F:\ProGram\Python_Base\Agent\08_skills\.claude\skills
          （也就是本文件所在的 Agent/08_skills/ 下面）

两个位置怎么选：

    | 位置 | 生效范围 | 适合放什么 |
    |---|---|---|
    | ~/.claude/skills/      | 本机所有项目 | 通用技能：文档处理、代码审查、爬虫… |
    | <项目>/.claude/skills/ | 只有这个项目 | 项目专属：本仓库的接口规范、数据字典… |

安装动作本质上就是「把一个技能目录复制过去」，没有任何注册步骤：

    # PowerShell（本机 shell）
    Copy-Item -Recurse -Force <技能目录> $env:USERPROFILE\.claude\skills\
    # bash（远程 Linux 服务器）
    cp -r <技能目录> ~/.claude/skills/

  本机全局路径的当前状态（只读列举，不做修改）：
    类型：目录联接（junction）  → \\?\F:\ProGramApp\DSH\skills
    已安装全局技能 32 个：['browser', 'deep-agents-core', 'deep-agents-memory', 'deep-agents-orchestration', 'deepagents-python-quickstart', 'dispatching-parallel-agents', 'docx', 'dsh-infra', 'ecosystem-primer', 'langchain-dependencies', 'langchain-fundamentals', 'langchain-middleware', 'langchain-python-quickstart', 'langchain-rag', 'langgraph-cli', 'langgraph-fundamentals', 'langgraph-human-in-the-loop', 'langgraph-persistence', 'langgraph-python-quickstart', 'lazy-senior-dev', 'managed-deep-agents', 'officecli', 'pdf', 'playwright', 'playwright-interactive', 'pptx', 'skill-install', 'tai-memory-stack', 'test-driven-development', 'verification-before-completion', 'writing-skills', 'xlsx']

  ★ 实测提醒：多个来源时的覆盖方向
      本机 deepagents 0.7.13 的 SkillsMiddleware 规则是「后面的来源覆盖前面同名的
      技能」（last one wins），并在系统提示词里把【最后一个】来源标成
      (higher priority)。所以代码里挂多个来源时，把项目级放【后面】：

          skills=["/skills/", "/.claude/skills/"]
                  ↑ 基础/全局            ↑ 项目级，优先级更高

      本节最后的验证环节会把这条规则跑出来给你看。
```

### 3.3 自动创建技能：四步 + skill-creator

`skill-creator` 替你做的事，其实就是上一课手工做的那几件：
建目录 → 起名字 → 写 front-matter（`name` / `description`）→ 写正文指令 →
必要时补 `scripts/` `references/` `assets/` → 自检 `name` 是否等于目录名。

一句话取舍：**一次性、临时用的提示词**直接写在对话里；
**反复要用、要给别人用、要版本管理的**做成技能。

In [ ]:
# ============================================================
# 3.3 自动创建技能：不用手写 front-matter
# ============================================================
def section_auto_create() -> None:
    print()
    print("=" * 78)
    print("2. 自动创建技能：不用手写 front-matter")
    print("=" * 78)
    print(
        """
课案原文四步：
    1. 下载 ComposioHQ/awesome-claude-skills
    2. 将当中的 skill-creator 放入 `~/.claude/skills/`
    3. 打开 PowerShell，输入 Claude
    4. 说：帮我创建一个 skill，使用 qwen3-vl 将流程图的图片转 mermaid

逐条展开成可以照抄的操作：

    # ① 拉取技能合集（ComposioHQ 维护的 awesome 列表，里面都是可用的现成技能）
    git clone https://github.com/ComposioHQ/awesome-claude-skills.git

    # ② 只把 skill-creator 这一个装进全局技能目录
    #    Windows / PowerShell
    Copy-Item -Recurse -Force .\\awesome-claude-skills\\skill-creator `
                $env:USERPROFILE\\.claude\\skills\\
    #    Linux / macOS
    cp -r ./awesome-claude-skills/skill-creator ~/.claude/skills/

    # ③ 在项目目录下启动 Claude Code
    claude

    # ④ 直接用自然语言提需求，skill-creator 会替你生成目录 + SKILL.md
    帮我创建一个 skill，使用 qwen3-vl 将流程图的图片转 mermaid

skill-creator 帮你做的事情，其实就是本仓库 02 小节手工做的那几件：
    建目录 → 起名字 → 写 front-matter（name/description）→ 写正文指令 →
    必要时补 scripts/ references/ assets/ → 自检 name 是否等于目录名。

一句话取舍：
    一次性、临时用的提示词 → 直接写在对话里；
    反复要用、要给别人用、要版本管理的 → 做成技能。

课案里那个 image-to-mermaid 技能（deepagents 小节也在用它）就是
「用 qwen3-vl 把流程图图片转成 mermaid」这个需求的产物：
    image-to-mermaid/
    ├── SKILL.md          # 告诉 agent「不要直接 read_file 读图片，去跑脚本」
    ├── scripts/          # 调 qwen3-vl 的转换脚本
    └── references/       # mermaid 语法规范（需要时才读）
课案给的分发包：https://dya123.oss-cn-beijing.aliyuncs.com/haydon-image-gen.zip
"""
    )


section_auto_create()

### 预期输出

```text

==============================================================================
2. 自动创建技能：不用手写 front-matter
==============================================================================

课案原文四步：
    1. 下载 ComposioHQ/awesome-claude-skills
    2. 将当中的 skill-creator 放入 `~/.claude/skills/`
    3. 打开 PowerShell，输入 Claude
    4. 说：帮我创建一个 skill，使用 qwen3-vl 将流程图的图片转 mermaid

逐条展开成可以照抄的操作：

    # ① 拉取技能合集（ComposioHQ 维护的 awesome 列表，里面都是可用的现成技能）
    git clone https://github.com/ComposioHQ/awesome-claude-skills.git

    # ② 只把 skill-creator 这一个装进全局技能目录
    #    Windows / PowerShell
    Copy-Item -Recurse -Force .\awesome-claude-skills\skill-creator `
                $env:USERPROFILE\.claude\skills\
    #    Linux / macOS
    cp -r ./awesome-claude-skills/skill-creator ~/.claude/skills/

    # ③ 在项目目录下启动 Claude Code
    claude

    # ④ 直接用自然语言提需求，skill-creator 会替你生成目录 + SKILL.md
    帮我创建一个 skill，使用 qwen3-vl 将流程图的图片转 mermaid

skill-creator 帮你做的事情，其实就是本仓库 02 小节手工做的那几件：
    建目录 → 起名字 → 写 front-matter（name/description）→ 写正文指令 →
    必要时补 scripts/ references/ assets/ → 自检 name 是否等于目录名。

一句话取舍：
    一次性、临时用的提示词 → 直接写在对话里；
    反复要用、要给别人用、要版本管理的 → 做成技能。

课案里那个 image-to-mermaid 技能（deepagents 小节也在用它）就是
「用 qwen3-vl 把流程图图片转成 mermaid」这个需求的产物：
    image-to-mermaid/
    ├── SKILL.md          # 告诉 agent「不要直接 read_file 读图片，去跑脚本」
    ├── scripts/          # 调 qwen3-vl 的转换脚本
    └── references/       # mermaid 语法规范（需要时才读）
课案给的分发包：https://dya123.oss-cn-beijing.aliyuncs.com/haydon-image-gen.zip
```

### 3.4 各平台对 Skills 的支持

⚠️ 一个容易混淆的点：**不同客户端读的目录不一定一样**。
Claude Code 认 `~/.claude/skills/` 与 `<项目>/.claude/skills/`；
Cursor / Trae / OpenCode 各有自己的约定目录。
但 `SKILL.md` 的**文件格式是同一套**（Agent Skills 规范：
YAML front-matter + Markdown 正文 + 可选的 `scripts` / `references` / `assets`），
所以同一个技能目录复制到不同客户端，通常都能直接用。

In [ ]:
# ============================================================
# 3.4 各平台对 Skills 的支持情况
# ============================================================
def section_platforms() -> None:
    print("=" * 78)
    print("3. 各平台对 Skills 的支持情况")
    print("=" * 78)
    print(
        """
课案原文：
    Claude Code、Cursor、Trae / OpenCode 均支持免费使用 Skills，
    其中 Claude Code 为官方推荐且功能最丰富。
"""
    )

    rows = [
        # 「定位」和「Skills 支持情况」两列都是从课案原文整理出来的：
        # 课案只说了「四家都支持，Claude Code 是官方推荐且功能最丰富」，
        # 这里把那句话拆成每家的具体形态，便于横向对照。
        ["Claude Code", "Anthropic 官方 CLI", "官方推荐，功能最丰富：\n自动触发、渐进式披露、可执行脚本"],
        ["Cursor", "AI 代码编辑器", "在编辑器里识别 SKILL.md，\n与补全/对话结合"],
        ["Trae", "AI IDE（字节）", "支持 Skills，免费使用"],
        ["OpenCode", "开源终端 AI 编码工具", "支持 Skills，免费使用"],
    ]
    headers = ["平台", "定位", "Skills 支持情况"]
    widths = [14, 26, 44]
    display = lambda s: sum(2 if ord(c) > 0x2E7F else 1 for c in s)  # noqa: E731
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(line)
    print("| " + " | ".join(h + " " * (w - display(h)) for h, w in zip(headers, widths)) + " |")
    print(line)
    for row in rows:
        cells = [c.split("\n") for c in row]
        for i in range(max(len(c) for c in cells)):
            parts = [
                (c[i] if i < len(c) else "") + " " * (w - display(c[i] if i < len(c) else ""))
                for c, w in zip(cells, widths)
            ]
            print("| " + " | ".join(parts) + " |")
        print(line)

    print(
        """
  ⚠ 一个容易混淆的点：不同客户端读的目录不一定一样。
        Claude Code 认 ~/.claude/skills/ 与 <项目>/.claude/skills/；
        Cursor / Trae / OpenCode 各有自己的约定目录。
    但 SKILL.md 的【文件格式】是同一套（Agent Skills 规范：
    YAML front-matter + Markdown 正文 + 可选的 scripts/references/assets），
    所以同一个技能目录复制到不同客户端，通常都能直接用。

  资源参考（课案给的几个入口）：
    官方技能示例、Skills 市场、Skills 市场 2、开源技能项目 ——
    找现成技能优先看这四类；自己写则参考本仓库 01/02 两节。
"""
    )


section_platforms()

### 预期输出

```text
==============================================================================
3. 各平台对 Skills 的支持情况
==============================================================================

课案原文：
    Claude Code、Cursor、Trae / OpenCode 均支持免费使用 Skills，
    其中 Claude Code 为官方推荐且功能最丰富。

+----------------+----------------------------+----------------------------------------------+
| 平台           | 定位                       | Skills 支持情况                              |
+----------------+----------------------------+----------------------------------------------+
| Claude Code    | Anthropic 官方 CLI         | 官方推荐，功能最丰富：                       |
|                |                            | 自动触发、渐进式披露、可执行脚本             |
+----------------+----------------------------+----------------------------------------------+
| Cursor         | AI 代码编辑器              | 在编辑器里识别 SKILL.md，                    |
|                |                            | 与补全/对话结合                              |
+----------------+----------------------------+----------------------------------------------+
| Trae           | AI IDE（字节）             | 支持 Skills，免费使用                        |
+----------------+----------------------------+----------------------------------------------+
| OpenCode       | 开源终端 AI 编码工具       | 支持 Skills，免费使用                        |
+----------------+----------------------------+----------------------------------------------+

  ⚠ 一个容易混淆的点：不同客户端读的目录不一定一样。
        Claude Code 认 ~/.claude/skills/ 与 <项目>/.claude/skills/；
        Cursor / Trae / OpenCode 各有自己的约定目录。
    但 SKILL.md 的【文件格式】是同一套（Agent Skills 规范：
    YAML front-matter + Markdown 正文 + 可选的 scripts/references/assets），
    所以同一个技能目录复制到不同客户端，通常都能直接用。

  资源参考（课案给的几个入口）：
    官方技能示例、Skills 市场、Skills 市场 2、开源技能项目 ——
    找现成技能优先看这四类；自己写则参考本仓库 01/02 两节。
```

### 3.5 真实安装演示：复制到项目级 `.claude/skills/`

源文件每次都「先 `rmtree` 再 `copytree`」覆盖安装目标。本仓库里
`Agent/08_skills/.claude/skills/code-review-skill/` **已经纳入版本管理**，
而且与 `skills/` 下的源技能逐字节相同 —— 每次运行都删除重建只会无谓地改动工作区。

所以这里加了一个开关：

- `FORCE_INSTALL = False`（默认）：目标已存在时走**只读比对**，不写任何文件；
- `FORCE_INSTALL = True`：走源文件原本的「先清理再复制」逻辑（新克隆的仓库第一次安装时用得上）。

安装动作的本质没有变，还是那三行：`mkdir` → `copytree` → 完事。

In [ ]:
# ============================================================
# 3.5 真实安装演示：复制到项目级 .claude/skills/
# ============================================================
# 默认只读：目标目录在版本管理里，不重复改写工作区。要真的重装就改成 True。
FORCE_INSTALL = False


def verify_installed() -> bool:
    """只读比对：源技能目录与项目级已安装目录是否逐文件一致（只比内容，不比时间戳）。"""
    src_files = sorted(p for p in SOURCE_SKILL.rglob("*") if p.is_file())
    ok = True
    for p in src_files:
        rel = p.relative_to(SOURCE_SKILL)
        dst = INSTALLED_SKILL / rel
        if not dst.exists() or dst.read_bytes() != p.read_bytes():
            print(f"    ✗ 不一致：{rel}")
            ok = False
    for q in sorted(p for p in INSTALLED_SKILL.rglob("*") if p.is_file()):
        if not (SOURCE_SKILL / q.relative_to(INSTALLED_SKILL)).exists():
            print(f"    ✗ 多出文件：{q.relative_to(INSTALLED_SKILL)}")
            ok = False
    if ok:
        print(f"    ✓ {len(src_files)} 个文件逐字节一致："
              f"{[str(p.relative_to(SOURCE_SKILL)) for p in src_files]}")
    return ok


def section_install() -> bool:
    print()
    print("=" * 78)
    print("4. 真实安装演示：复制到项目级 .claude/skills/")
    print("=" * 78)

    if not SOURCE_SKILL.exists():
        print(f"  ！源技能不存在：{SOURCE_SKILL}")
        print("  请先运行：uv run Agent/08_skills/02_SKILL示例_jxsd.py")
        return False

    print("  ── 安装【前】──")
    # 只列 08_skills 下的两个载体目录，跳过 tmp_nb_work / __pycache__ 这类运行产物 ——
    # 否则目录树里会混进其它 notebook 的临时文件，输出不可复现。
    print("  08_skills 下的一级目录：",
          sorted(p.name for p in HERE.iterdir()
                 if p.is_dir() and p.name not in ("tmp_nb_work", "__pycache__")))
    print_tree(SKILLS_ROOT, "源技能父目录 skills/（skills 参数要指向的那一层）")
    print()

    if FORCE_INSTALL or not INSTALLED_SKILL.exists():
        # 幂等：目标已存在就先删掉再复制，避免 copytree 报 FileExistsError，
        # 也避免新旧文件混在一起（技能是整体替换的，不增量合并）。
        if INSTALLED_SKILL.exists():
            shutil.rmtree(INSTALLED_SKILL)
            print(f"  目标已存在，先清理：{INSTALLED_SKILL}")
        PROJECT_CLAUDE_DIR.mkdir(parents=True, exist_ok=True)

        # copytree：把整个技能目录（含 references/）原样搬过去
        shutil.copytree(SOURCE_SKILL, INSTALLED_SKILL)
        print(f"  已复制 {SOURCE_SKILL.name}/ → {INSTALLED_SKILL.relative_to(HERE.parent.parent)}")
    else:
        print(f"  目标已存在（仓库已跟踪）：{INSTALLED_SKILL}")
        print("  → 本 notebook 只做【只读比对】，不删除重建（FORCE_INSTALL=False）")
        verify_installed()
    print()
    print("  ⚠ 注意：本演示只装【项目级】路径（08_skills/.claude/skills/）。")
    print(f"    全局路径 {GLOBAL_CLAUDE_SKILLS} 属于用户级配置，本文件不写入。")
    print("    要装全局，手动执行：")
    print("      Copy-Item -Recurse -Force Agent\\08_skills\\skills\\code-review-skill "
          "$env:USERPROFILE\\.claude\\skills\\")
    print()

    print("  ── 安装【后】──")
    print_tree(PROJECT_CLAUDE_DIR.parent, ".claude 目录（项目级安装位置）")
    return True


if section_install():
    print("  → 安装路径已就绪；3.6 会把两个来源分别挂上，验证技能真的能被发现。")

### 预期输出

```text

==============================================================================
4. 真实安装演示：复制到项目级 .claude/skills/
==============================================================================
  ── 安装【前】──
  08_skills 下的一级目录： ['.claude', 'skills']
  【源技能父目录 skills/（skills 参数要指向的那一层）】F:\ProGram\Python_Base\Agent\08_skills\skills
    └── code-review-skill/
        ├── references/
        │   ├── javascript_rules.md    # 54 行
        │   └── python_rules.md    # 58 行
        └── SKILL.md    # 37 行

  目标已存在（仓库已跟踪）：F:\ProGram\Python_Base\Agent\08_skills\.claude\skills\code-review-skill
  → 本 notebook 只做【只读比对】，不删除重建（FORCE_INSTALL=False）
    ✓ 3 个文件逐字节一致：['references\\javascript_rules.md', 'references\\python_rules.md', 'SKILL.md']

  ⚠ 注意：本演示只装【项目级】路径（08_skills/.claude/skills/）。
    全局路径 C:\Users\QianXi\.claude\skills 属于用户级配置，本文件不写入。
    要装全局，手动执行：
      Copy-Item -Recurse -Force Agent\08_skills\skills\code-review-skill $env:USERPROFILE\.claude\skills\

  ── 安装【后】──
  【.claude 目录（项目级安装位置）】F:\ProGram\Python_Base\Agent\08_skills\.claude
    └── skills/
        └── code-review-skill/
            ├── references/
            │   ├── javascript_rules.md    # 54 行
            │   └── python_rules.md    # 58 行
            └── SKILL.md    # 37 行
  → 安装路径已就绪；3.6 会把两个来源分别挂上，验证技能真的能被发现。
```

### 3.6 验证：技能是否真的可被发现，以及多来源的覆盖方向

把两个目录分别挂上、再一起挂上，看发现结果与来源标签 ——
这就是「复制过去就算装好」和「数组里谁在后面谁生效」的实测。

In [ ]:
# ============================================================
# 3.6 验证：技能是否真的可被发现，以及多来源的覆盖方向
# ============================================================
def section_verify() -> None:
    print()
    print("=" * 78)
    print("5. 验证：技能是否真的可被发现，以及多来源的覆盖方向")
    print("=" * 78)

    from deepagents.backends.local_shell import LocalShellBackend
    from deepagents.middleware.skills import SkillsMiddleware

    backend = LocalShellBackend(root_dir=str(HERE))

    for sources in (["/skills/"], ["/.claude/skills/"], ["/skills/", "/.claude/skills/"]):
        middleware = SkillsMiddleware(backend=backend, sources=sources)
        update = middleware.before_agent({}, None, None)  # type: ignore[arg-type]
        found = [s["name"] for s in (update["skills_metadata"] if update else [])]
        print(f"  sources={sources}")
        print(f"    来源标签 = {middleware.source_labels}   ← 渲染成系统提示词里的 **xxx Skills**")
        print(f"    发现技能 = {found}")

    print(
        """
  结论：
    1. 两个目录都能被独立识别，复制过去就算安装完成 —— 没有注册表、没有配置文件；
    2. 一起挂上时，两个来源里同名的 code-review-skill 只会留一个 ——
       后一个来源（项目级 .claude/skills/）覆盖前一个（skills/），
       系统提示词里也会把最后一个标成 (higher priority)；
    3. 所以「改哪个目录生效」这件事，在框架层面取决于数组顺序 ——
       把想生效的那个放最后。
"""
    )


if READY["deepagents"]:
    section_verify()
else:
    print("（跳过 3.6：deepagents 未安装）")

### 预期输出

```text

==============================================================================
5. 验证：技能是否真的可被发现，以及多来源的覆盖方向
==============================================================================
  sources=['/skills/']
    来源标签 = ['Skills']   ← 渲染成系统提示词里的 **xxx Skills**
    发现技能 = ['code-review-skill']
  sources=['/.claude/skills/']
    来源标签 = ['Claude']   ← 渲染成系统提示词里的 **xxx Skills**
    发现技能 = ['code-review-skill']
  sources=['/skills/', '/.claude/skills/']
    来源标签 = ['Skills', 'Claude']   ← 渲染成系统提示词里的 **xxx Skills**
    发现技能 = ['code-review-skill']

  结论：
    1. 两个目录都能被独立识别，复制过去就算安装完成 —— 没有注册表、没有配置文件；
    2. 一起挂上时，两个来源里同名的 code-review-skill 只会留一个 ——
       后一个来源（项目级 .claude/skills/）覆盖前一个（skills/），
       系统提示词里也会把最后一个标成 (higher priority)；
    3. 所以「改哪个目录生效」这件事，在框架层面取决于数组顺序 ——
       把想生效的那个放最后。
```

### 3.7 小结（源文件的最后一节）：装一个技能要做的三件事

1. 准备好一个合规的技能目录（`SKILL.md` 的 `name` 必须等于目录名）；
2. 复制到加载路径之一（全局或项目级）；
3. 重启客户端（Claude Code / Cursor / Trae / OpenCode），技能清单会在启动时被扫描一次。

本节在磁盘上留下的东西（受版本管理时建议把 `.claude/skills/` 一起提交，
这样团队里每个人 clone 下来就自带项目技能）；全局路径本次**未做任何写入**。

In [ ]:
# ============================================================
# 3.7 小结：装一个技能要做的三件事
# ============================================================
def section_summary() -> None:
    print("=" * 78)
    print("6. 小结：装一个技能要做的三件事")
    print("=" * 78)
    print(
        f"""
    1. 准备好一个合规的技能目录（SKILL.md 的 name 必须等于目录名）；
    2. 复制到加载路径之一：
           全局  {GLOBAL_CLAUDE_SKILLS}
           项目  {PROJECT_CLAUDE_DIR}
    3. 重启客户端（Claude Code / Cursor / Trae / OpenCode），
       技能清单会在启动时被扫描一次。

    本节在磁盘上留下的东西（受版本管理时建议把 .claude/skills/ 一起提交，
    这样团队里每个人 clone 下来就自带项目技能）：
       {INSTALLED_SKILL}

    全局路径本次【未做任何写入】。
"""
    )


section_summary()

### 预期输出

```text
==============================================================================
6. 小结：装一个技能要做的三件事
==============================================================================

    1. 准备好一个合规的技能目录（SKILL.md 的 name 必须等于目录名）；
    2. 复制到加载路径之一：
           全局  C:\Users\QianXi\.claude\skills
           项目  F:\ProGram\Python_Base\Agent\08_skills\.claude\skills
    3. 重启客户端（Claude Code / Cursor / Trae / OpenCode），
       技能清单会在启动时被扫描一次。

    本节在磁盘上留下的东西（受版本管理时建议把 .claude/skills/ 一起提交，
    这样团队里每个人 clone 下来就自带项目技能）：
       F:\ProGram\Python_Base\Agent\08_skills\.claude\skills\code-review-skill

    全局路径本次【未做任何写入】。
```

## 4. 三种载体横向对照（收口）

把三节跑完的结果收成一张表 —— 这也是本课的验收口径：

| 维度 | ① `skills=[...]` 本地目录 | ② 云技能（Store） | ③ Claude Code `.claude/skills/` |
|---|---|---|---|
| **存哪** | 宿主机目录 `Agent/08_skills/skills/` | LangGraph Store（本机 PostgreSQL `langgraph` 库的 `store` 表） | 宿主机目录：全局 `~/.claude/skills/` 或项目 `<项目>/.claude/skills/` |
| **谁发现** | DeepAgents 的 `SkillsMiddleware`（传了 `skills=` 自动挂） | 同一个 `SkillsMiddleware` | Claude Code / Cursor / Trae / OpenCode 客户端自己扫 |
| **怎么加载** | `sources=["/skills/"]` → 启动时 ls 一层，只把 `name`+`description` 拼进 system prompt；模型需要时 `read_file` 拉全文 | 写法不变，只是 backend 换成 `CompositeBackend`，`/skills/` 前缀路由到 `StoreBackend`；文件从数据库读 | 启动时扫目录得清单；「安装」= `copytree` 过去，**没有注册表** |
| **隔离粒度** | 机器 / 目录（没有隔离） | **`namespace`**（`("user-xxx",)`），一套服务给所有人用 | 全局（所有项目）/ 项目级（只有这个项目） |
| **改一次生效范围** | 该目录下所有使用者 | 该 namespace 下的使用者 | 全局装 → 全机；项目装 → 全项目 |
| **适用场景** | 单机、单用户；技能跟代码仓库一起走 | 多用户 SaaS：按登录用户发技能库 | 把技能装进真实 IDE / CLI 给同事用 |
| **本课实测结论** | 传 `["/skills/"]` 发现 1 个；传具体技能目录 / `SKILL.md` / `["/"]` 都是 0 个 | 同一份代码只换 namespace，`user-001` 看到 2 个、`user-002` 看到 1 个且互不可见 | 两个目录都能独立发现；一起挂时**数组里靠后的来源生效** |

**选型建议**：

1. 只想让 agent 会用某个流程 → 做成技能目录放进仓库，`skills=["/skills/"]`（最快）。
2. 一套服务要服务多个用户、每人技能不同 → 上 Store，把 `namespace` 接到鉴权信息上。
3. 想让技能在 IDE / CLI 里自动触发 → 复制到 `.claude/skills/`（项目级更推荐，能进版本管理）。

三者**不互斥**：同一份 `SKILL.md` 可以既放在仓库目录里给 DeepAgents 用、
又复制一份进 `.claude/skills/` 给 Claude Code 用，还能再灌进 Store 做云端分发。

## 小结

- **技能 = 一个目录 + `SKILL.md`**；三种载体只是「目录放在哪、谁去扫」的区别，
  文件格式同一套（YAML front-matter + Markdown 正文 + 可选 `scripts`/`references`/`assets`）。
- **`skills` 参数指向技能的父目录**：写具体技能目录 / `SKILL.md` / `["/"]`（而技能不在根下）
  都是**发现 0 个**。中间件的规则是「ls 这一层 → 每个一级子目录找 `SKILL.md`」。
- **渐进式披露**：常驻 system prompt 的只有 `name` + `description` + 路径；
  正文和 `references/` 是模型自己 `read_file` 拉进来的 —— 这才是省 Token 的落点。
- **云技能不换写法、只换 backend**：`CompositeBackend(default=StateBackend(), routes={"/skills/": StoreBackend(...)})`，
  隔离粒度是 `namespace`，不是机器也不是目录。
- **安装技能 = 复制目录**：没有注册表、没有配置文件；全局 `~/.claude/skills/`、
  项目 `<项目>/.claude/skills/`，重启客户端生效。
- **多来源覆盖方向 = 数组顺序**：`SkillsMiddleware` 里后面的来源覆盖前面同名的技能，
  系统提示词把最后一个标成 `(higher priority)`。

## 常见坑

1. **`skills` 写成具体技能目录或 `SKILL.md` 文件** → 发现 0 个技能，而且**不报错**
   （写 `SKILL.md` 会在 `before_agent` 时打一条 `not_a_directory` 警告）。
   排查第一步：把它改成**技能的父目录**。
2. **`skills=["/"]` 就一定是 0 个吗？不一定** —— 取决于 backend 根目录下是否**直接**
   摆着技能目录。本仓库技能在 `skills/` 下面一层，所以传 `/` 是 0 个；
   课案里 `root_dir="."` 且技能就放根目录，`["/"]` 才成立。**路径是 backend 的虚拟路径，不是宿主机路径。**
3. **云技能的 `store.put` key 带上了路由前缀** → 虚拟路径变成
   `/skills/skills/sql-gen/SKILL.md`，`skills=["/skills/"]` 一个都发现不了。
   `CompositeBackend` 命中路由后会把前缀**剥掉**，所以 key 要相对路由前缀。
4. **`thread_id` 写死** → 第二次运行命中上一轮的检查点，而 `SkillsMiddleware` 的逻辑是
   「state 里已有 `skills_metadata` 就不再加载」，于是技能加载被静默跳过、行为时好时坏。
   教学 / 测试脚本用 `uuid`。
5. **`LocalShellBackend(root_dir=...)` 换了，`skills` 里的路径没跟着换** → 同一个坑：
   虚拟 `/skills/` 落到别的地方，发现 0 个。
6. **`inherited_env` / `inherit_env` 忘了开** → 技能里的脚本拿不到 `.env` 里的 API Key，
   表现为「脚本能跑但调用模型 401」。
7. **往 `~/.claude/skills/` 写东西前先确认它是不是目录联接（junction）**：
   Windows 上 `is_symlink()` 返回 `False`，要用 `os.path.isjunction()` 判断 ——
   否则你会在源的仓库里原地改文件。
8. **`.claude/skills/` 每次运行都删除重建** → 已经在版本管理里的目录会被无谓改写。
   要么只读比对（本 notebook 的做法），要么把它 `gitignore` 掉。

## 官方链接

- **Skills（DeepAgents，本节主角）**：<https://docs.langchain.com/oss/python/deepagents/skills>
- Backends（`LocalShellBackend` / `CompositeBackend` / `StoreBackend`）：
  <https://docs.langchain.com/oss/python/deepagents/backends>
- Stores（Store 的 `namespace`、跨会话持久化）：<https://docs.langchain.com/oss/python/langgraph/stores>
- Persistence（`checkpointer` 与 `store` 的分工）：
  <https://docs.langchain.com/oss/python/langgraph/persistence>
- Claude Code Skills（目录约定与安装）：<https://code.claude.com/docs/en/skills>
- 课案提到的技能合集（自动创建技能用）：<https://github.com/ComposioHQ/awesome-claude-skills>